In [2]:
import sys
import pandas as pd

sys.path.append("/var/www/python/Qingcheng/nighthawk/")
from nighthawk.util import sql_functions
from nighthawk.data.pipeline.common_functions.load import Load
from nighthawk.data.pipeline.common_functions.wind import Wind
from nighthawk.data.pipeline.common_functions.solar import Solar
from nighthawk.data.pipeline.common_functions.genoutage import GenOutage
from nighthawk.data.pipeline.var_handler.fuel_type import FuelType

START = "2026-03-23"
END   = "2026-04-10"
BAA_ZONES = ["E", "W"]

load     = Load("SPP")
wind     = Wind("SPP")
solar    = Solar("SPP")
genout   = GenOutage("SPP")
fueltype = FuelType("SPP")
print(f"Objects created for SPP  |  {START} → {END}")

PERIODS = [
    ("2026-03-23", "2026-03-28", "3/23-3/28"),
    ("2026-04-05", "2026-04-10", "4/5-4/10"),
]

def check_df(df, period, impute, pivot=None, extra_label=""):
    null_count = df.isnull().sum().sum()
    num_df = df.select_dtypes(include="number")
    zero_count = (num_df == 0).sum().sum()
    pivot_str = f", pivot={pivot}" if pivot is not None else ""
    extra_str = f" [{extra_label}]" if extra_label else ""
    print(f"  [{period}]{extra_str} impute={impute}{pivot_str} "
          f"→ Shape={df.shape}, Nulls={null_count}, Zeros={zero_count}")
    if impute and null_count > 0:
        null_mask = df.isnull().any(axis=1)
        print(f"  ⚠️  WARNING: impute=True but {null_count} null values remain!")
        print("  Top 5 rows with nulls:")
        display(df[null_mask].head(5))

print("PERIODS and check_df helper loaded.")


Objects created for SPP  |  2026-03-23 → 2026-04-10
PERIODS and check_df helper loaded.


## 1. Total Load

In [13]:
print("=== 1. Total Load ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_total = load.get_total_load(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_total, period, impute)
display(df_total.head())


=== 1. Total Load ===
  [3/23-3/28] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 4), Nulls=0, Zeros=0


,dt,hr,spp_load_total_forecast_f,spp_load_total_actual_a
0,2026-04-05,1,31403.0,30316.0
1,2026-04-05,2,30814.0,29499.0
2,2026-04-05,3,30536.0,29276.0
3,2026-04-05,4,30499.0,29501.0
4,2026-04-05,5,30692.0,29730.0


## 2. BAA Zonal Load (E and W)

In [14]:
print("=== 2. BAA Zonal Load ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_baa = load.get_baa_zonal_load(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_baa, period, impute, pivot)
display(df_baa.head())


=== 2. BAA Zonal Load ===
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=True, pivot=False → Shape=(144, 5), Nulls=0, Zeros=0
  [3/23-3/28] impute=False, pivot=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=False, pivot=False → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=True → Shape=(144, 6), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=False → Shape=(288, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=True → Shape=(144, 6), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=False → Shape=(288, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_baa_zonal_load_forecast_f,spp_baa_zonal_load_actual_a
0,2026-04-05,1,E,28845.0,27833.0
1,2026-04-05,1,W,2558.0,2483.0
2,2026-04-05,2,E,28377.0,27407.0
3,2026-04-05,2,W,2437.0,2092.0
4,2026-04-05,3,E,28141.0,27224.0


## 3. Broadzonal Load

In [15]:
print("=== 3. Broadzonal Load ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_bz = load.get_broadzonal_load(s, e, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_bz, period, impute, pivot)
display(df_bz.head())


=== 3. Broadzonal Load ===
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 8), Nulls=0, Zeros=48
  [3/23-3/28] impute=True, pivot=False → Shape=(432, 5), Nulls=0, Zeros=48
  [3/23-3/28] impute=False, pivot=True → Shape=(144, 8), Nulls=0, Zeros=48
  [3/23-3/28] impute=False, pivot=False → Shape=(432, 5), Nulls=0, Zeros=48
  [4/5-4/10] impute=True, pivot=True → Shape=(144, 8), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=False → Shape=(432, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=True → Shape=(144, 8), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=False → Shape=(432, 5), Nulls=0, Zeros=0


,dt,hr,broadzone,spp_broadzonal_load_forecast_f,spp_broadzonal_load_actual_a
0,2026-04-05,1,NORTH,7577.24,7909.83
1,2026-04-05,1,SOUTH,20680.77,19923.30
2,2026-04-05,1,WEST,2482.78,2482.78
3,2026-04-05,2,NORTH,7596.35,7842.32
4,2026-04-05,2,SOUTH,20407.25,19564.45


## 4. Zonal Load

In [16]:
print("=== 4. Zonal Load ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_zonal = load.get_zonal_load(s, e, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_zonal, period, impute, pivot)
# Extra: show_warnings (data unchanged, suppresses internal prints)
for s, e, period in PERIODS:
    df_sw = load.get_zonal_load(s, e, var_spec=["f", "a"], show_warnings=False)
    check_df(df_sw, period, True, extra_label="show_warnings=False")
display(df_zonal.head())


=== 4. Zonal Load ===


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/load.py:1601: UserWarning: No zonal load forecast available for zone(s) ['PRPA', 'WACM', 'WAUW']; using actual load as forecast.
  warnings.warn(


  [3/23-3/28] impute=True, pivot=True → Shape=(144, 40), Nulls=96, Zeros=0
  ⚠️  WARNING: impute=True but 96 null values remain!
  Top 5 rows with nulls:


,dt,hr,csws_spp_zonal_load_actual_a,ede_spp_zonal_load_actual_a,grda_spp_zonal_load_actual_a,indn_spp_zonal_load_actual_a,kacy_spp_zonal_load_actual_a,kcpl_spp_zonal_load_actual_a,les_spp_zonal_load_actual_a,mps_spp_zonal_load_actual_a,...,okge_spp_zonal_load_forecast_f,oppd_spp_zonal_load_forecast_f,seci_spp_zonal_load_forecast_f,sprm_spp_zonal_load_forecast_f,sps_spp_zonal_load_forecast_f,wacm_spp_zonal_load_forecast_f,waue_spp_zonal_load_forecast_f,wauw_spp_zonal_load_forecast_f,wfec_spp_zonal_load_forecast_f,wr_spp_zonal_load_forecast_f
0,2026-03-23,1,4270.28,433.37,991.85,74.38,199.40,1398.51,386.15,712.73,...,3661.70,1607.17,643.91,265.49,3985.84,NaN,3928.44,NaN,921.72,2807.58
1,2026-03-23,2,4057.49,419.70,985.55,72.22,194.56,1379.42,383.56,703.39,...,3620.35,1593.58,650.76,261.48,3905.23,NaN,3927.27,NaN,906.52,2793.62
2,2026-03-23,3,3966.18,418.10,988.02,71.21,197.98,1380.38,382.00,708.44,...,3560.17,1606.56,647.49,262.13,3926.21,NaN,3929.56,NaN,905.11,2824.29
3,2026-03-23,4,3915.02,420.08,994.88,71.81,197.73,1399.93,388.38,724.40,...,3565.55,1619.91,663.38,266.48,3940.77,NaN,3942.05,NaN,912.67,2882.12
4,2026-03-23,5,3948.25,428.40,998.12,75.25,197.07,1446.30,402.40,760.03,...,3673.10,1654.83,652.84,284.34,3981.87,NaN,3947.04,NaN,952.06,2996.67


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/load.py:1601: UserWarning: No zonal load forecast available for zone(s) ['PRPA', 'WACM', 'WAUW']; using actual load as forecast.
  warnings.warn(


...

/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/load.py:1601: UserWarning: No zonal load forecast available for zone(s) ['PRPA', 'WACM', 'WAUW']; forecast will be NaN.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/load.py:1601: UserWarning: No zonal load forecast available for zone(s) ['PRPA', 'WACM', 'WAUW']; forecast will be NaN.
  warnings.warn(


  [4/5-4/10] impute=False, pivot=False → Shape=(2880, 5), Nulls=432, Zeros=0
  [3/23-3/28] [show_warnings=False] impute=True → Shape=(2736, 5), Nulls=96, Zeros=0
  ⚠️  WARNING: impute=True but 96 null values remain!
  Top 5 rows with nulls:


,dt,hr,zone,spp_zonal_load_forecast_f,spp_zonal_load_actual_a
14,2026-03-23,1,WACM,NaN,NaN
16,2026-03-23,1,WAUW,NaN,NaN
33,2026-03-23,2,WACM,NaN,NaN
35,2026-03-23,2,WAUW,NaN,NaN
52,2026-03-23,3,WACM,NaN,NaN


  [4/5-4/10] [show_warnings=False] impute=True → Shape=(2880, 5), Nulls=0, Zeros=0


,dt,hr,zone,spp_zonal_load_forecast_f,spp_zonal_load_actual_a
0,2026-04-05,1,CSWS,4282.80,3965.04
1,2026-04-05,1,EDE,472.04,448.18
2,2026-04-05,1,GRDA,940.73,955.33
3,2026-04-05,1,INDN,80.62,79.30
4,2026-04-05,1,KACY,204.43,193.66


## 5. Reserve Zonal Load

In [17]:
print("=== 5. Reserve Zonal Load ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_res = load.get_res_zonal_load(s, e, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_res, period, impute, pivot)
display(df_res.head())


=== 5. Reserve Zonal Load ===
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 14), Nulls=48, Zeros=0
  ⚠️  WARNING: impute=True but 48 null values remain!
  Top 5 rows with nulls:


,dt,hr,rz1_spp_res_zonal_load_actual_a,rz2_spp_res_zonal_load_actual_a,rz3_spp_res_zonal_load_actual_a,rz4_spp_res_zonal_load_actual_a,rz5_spp_res_zonal_load_actual_a,rz21_spp_res_zonal_load_actual_a,rz1_spp_res_zonal_load_forecast_f,rz2_spp_res_zonal_load_forecast_f,rz3_spp_res_zonal_load_forecast_f,rz4_spp_res_zonal_load_forecast_f,rz5_spp_res_zonal_load_forecast_f,rz21_spp_res_zonal_load_forecast_f
0,2026-03-23,1,3747.64,622.48,4146.80,15527.70,3930.17,NaN,3755.92,643.91,3985.84,15953.90,3928.44,NaN
1,2026-03-23,2,3738.15,624.73,4092.53,15067.55,3918.26,NaN,3728.84,650.76,3905.23,15675.63,3927.27,NaN
2,2026-03-23,3,3795.17,625.68,4074.49,14914.82,3932.70,NaN,3744.93,647.49,3926.21,15561.22,3929.56,NaN
3,2026-03-23,4,3821.75,633.51,4036.33,14887.56,3937.95,NaN,3800.74,663.38,3940.77,15687.26,3942.05,NaN
4,2026-03-23,5,3896.65,631.35,4046.50,15104.81,3930.84,NaN,3898.30,652.84,3981.87,16266.89,3947.04,NaN


  [3/23-3/28] impute=True, pivot=False → Shape=(864, 5), Nulls=48, Zeros=0
  ⚠️  WARNING: impute=True but 48 null values remain!
  Top 5 rows with nulls:


,dt,hr,ReserveZone,spp_res_zonal_load_forecast_f,spp_res_zonal_load_actual_a
5,2026-03-23,1,21,NaN,NaN
11,2026-03-23,2,21,NaN,NaN
17,2026-03-23,3,21,NaN,NaN
23,2026-03-23,4,21,NaN,NaN
29,2026-03-23,5,21,NaN,NaN


  [3/23-3/28] impute=False, pivot=True → Shape=(144, 13), Nulls=24, Zeros=0
  [3/23-3/28] impute=False, pivot=False → Shape=(864, 5), Nulls=168, Zeros=0
  [4/5-4/10] impute=True, pivot=True → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=False → Shape=(864, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=True → Shape=(144, 13), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=False → Shape=(864, 5), Nulls=144, Zeros=0


,dt,hr,ReserveZone,spp_res_zonal_load_forecast_f,spp_res_zonal_load_actual_a
0,2026-04-05,1,1,3706.70,3964.94
1,2026-04-05,1,2,641.73,605.52
2,2026-04-05,1,3,4048.47,4287.22
3,2026-04-05,1,4,15990.57,15030.56
4,2026-04-05,1,5,3870.54,3944.89


## 6. Load Error

In [18]:
print("=== 6. Load Error ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_le = load.get_load_error(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_le, period, impute)
# Extra: show_warnings
for s, e, period in PERIODS:
    df_sw = load.get_load_error(s, e, var_spec=["f", "a"], show_warnings=False)
    check_df(df_sw, period, True, extra_label="show_warnings=False")
display(df_le.head())


=== 6. Load Error ===
  [3/23-3/28] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] [show_warnings=False] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] [show_warnings=False] impute=True → Shape=(144, 4), Nulls=0, Zeros=0


,dt,hr,spp_load_error_f,spp_load_error_a
0,2026-04-05,1,-176.3400,-1087.0
1,2026-04-05,2,-12.2695,-1315.0
2,2026-04-05,3,32.9108,-1260.0
3,2026-04-05,4,38.1969,-998.0
4,2026-04-05,5,42.3038,-962.0


## 7. BAA Zonal Load Error (E and W)

In [46]:
print("=== 7. BAA Zonal Load Error ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_be = load.get_baa_zonal_load_error(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], impute=impute)
        check_df(df_be, period, impute)
        display(df_be[:20])
# Extra: show_warnings
for s, e, period in PERIODS:
    df_sw = load.get_baa_zonal_load_error(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], show_warnings=False)
    check_df(df_sw, period, True, extra_label="show_warnings=False")
display(df_be.head())


=== 7. BAA Zonal Load Error ===
  [3/23-3/28] impute=True → Shape=(144, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_baa_zonal_load_error_a,spp_baa_zonal_load_error_f
0,2026-03-23,1,E,23.0,172.9820
1,2026-03-23,2,E,1.0,228.0790
2,2026-03-23,3,E,103.0,274.0590
3,2026-03-23,4,E,-4.0,296.3350
4,2026-03-23,5,E,-139.0,270.0210
5,2026-03-23,6,E,-165.0,531.8810
6,2026-03-23,7,E,-29.0,610.1430
7,2026-03-23,8,E,157.0,335.8900
8,2026-03-23,9,E,314.0,112.7210
9,2026-03-23,10,E,388.0,221.6000


  [3/23-3/28] impute=False → Shape=(144, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_baa_zonal_load_error_a,spp_baa_zonal_load_error_f
0,2026-03-23,1,E,23.0,172.9820
1,2026-03-23,2,E,1.0,228.0790
2,2026-03-23,3,E,103.0,274.0590
3,2026-03-23,4,E,-4.0,296.3350
4,2026-03-23,5,E,-139.0,270.0210
5,2026-03-23,6,E,-165.0,531.8810
6,2026-03-23,7,E,-29.0,610.1430
7,2026-03-23,8,E,157.0,335.8900
8,2026-03-23,9,E,314.0,112.7210
9,2026-03-23,10,E,388.0,221.6000


  [4/5-4/10] impute=True → Shape=(288, 5), Nulls=0, Zeros=0


...

  [4/5-4/10] impute=False → Shape=(288, 5), Nulls=144, Zeros=0


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/load.py:2163: UserWarning: No model-predicted BAA-zonal load error forecast available for zone(s) ['W']; forecast will be NaN.
  warnings.warn(


,dt,hr,baa_zone,spp_baa_zonal_load_error_a,spp_baa_zonal_load_error_f
0,2026-04-05,1,E,-1012.0,-176.3400
1,2026-04-05,1,W,-75.0,NaN
2,2026-04-05,2,E,-970.0,-12.2695
3,2026-04-05,2,W,-345.0,NaN
4,2026-04-05,3,E,-917.0,32.9108
5,2026-04-05,3,W,-343.0,NaN
6,2026-04-05,4,E,-900.0,38.1969
7,2026-04-05,4,W,-98.0,NaN
8,2026-04-05,5,E,-934.0,42.3038
9,2026-04-05,5,W,-28.0,NaN


  [3/23-3/28] [show_warnings=False] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] [show_warnings=False] impute=True → Shape=(288, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_baa_zonal_load_error_a,spp_baa_zonal_load_error_f
0,2026-04-05,1,E,-1012.0,-176.3400
1,2026-04-05,1,W,-75.0,NaN
2,2026-04-05,2,E,-970.0,-12.2695
3,2026-04-05,2,W,-345.0,NaN
4,2026-04-05,3,E,-917.0,32.9108


---
## WIND

### 8. Total Wind

In [4]:
print("=== 8. Total Wind ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_wt = wind.get_total_wind(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_wt, period, impute)
# Extra: add_curtail_to_actual / cut_curtail_from_forecast
for s, e, period in PERIODS:
    for acc, ccf, lbl in [(True, False, "acc=T,ccf=F"), (False, True, "acc=F,ccf=T"), (True, True, "acc=T,ccf=T")]:
        df_c = wind.get_total_wind(s, e, var_spec=["f", "a"], add_curtail_to_actual=acc, cut_curtail_from_forecast=ccf)
        check_df(df_c, period, True, extra_label=lbl)
display(df_wt.head())


=== 8. Total Wind ===
  [3/23-3/28] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] [acc=T,ccf=F] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] [acc=F,ccf=T] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] [acc=T,ccf=T] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] [acc=T,ccf=F] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] [acc=F,ccf=T] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] [acc=T,ccf=T] impute=True → Shape=(144, 4), Nulls=0, Zeros=0


,dt,hr,spp_wind_total_forecast_f,spp_wind_total_actual_a
0,2026-04-05,1,9193.42,7587.80
1,2026-04-05,2,8872.17,7707.27
2,2026-04-05,3,8575.91,7838.81
3,2026-04-05,4,8316.86,7016.76
4,2026-04-05,5,8150.53,6713.68


### 9. BAA Zonal Wind (E and W)

In [5]:
print("=== 9. BAA Zonal Wind ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_wb = wind.get_baa_zonal_wind(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_wb, period, impute, pivot)
# Extra: add_curtail_to_actual / cut_curtail_from_forecast
for s, e, period in PERIODS:
    for acc, ccf, lbl in [(True, False, "acc=T,ccf=F"), (False, True, "acc=F,ccf=T"), (True, True, "acc=T,ccf=T")]:
        df_c = wind.get_baa_zonal_wind(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], add_curtail_to_actual=acc, cut_curtail_from_forecast=ccf)
        check_df(df_c, period, True, extra_label=lbl)
display(df_wb.head())


=== 9. BAA Zonal Wind ===
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=True, pivot=False → Shape=(144, 5), Nulls=0, Zeros=0
  [3/23-3/28] impute=False, pivot=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=False, pivot=False → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=True → Shape=(144, 6), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=False → Shape=(288, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=True → Shape=(144, 6), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=False → Shape=(288, 5), Nulls=0, Zeros=0


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(


  [3/23-3/28] [acc=T,ccf=F] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [3/23-3/28] [acc=F,ccf=T] impute=True → Shape=(144, 5), Nulls=0, Zeros=0


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(


  [3/23-3/28] [acc=T,ccf=T] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] [acc=T,ccf=F] impute=True → Shape=(288, 5), Nulls=0, Zeros=0


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(


  [4/5-4/10] [acc=F,ccf=T] impute=True → Shape=(288, 5), Nulls=0, Zeros=0
  [4/5-4/10] [acc=T,ccf=T] impute=True → Shape=(288, 5), Nulls=0, Zeros=0


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(


,dt,hr,baa_zone,spp_baa_zonal_wind_forecast_f,spp_baa_zonal_wind_actual_a
0,2026-04-05,1,E,8944.43,7355.50
1,2026-04-05,1,W,248.99,232.30
2,2026-04-05,2,E,8686.48,7575.86
3,2026-04-05,2,W,185.69,131.41
4,2026-04-05,3,E,8432.62,7782.39


### 10. Reserve Zonal Wind

In [6]:
print("=== 10. Reserve Zonal Wind ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_wr = wind.get_res_zonal_wind(s, e, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_wr, period, impute, pivot)
display(df_wr.head())


=== 10. Reserve Zonal Wind ===
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 12), Nulls=0, Zeros=0
  [3/23-3/28] impute=True, pivot=False → Shape=(720, 5), Nulls=0, Zeros=0
  [3/23-3/28] impute=False, pivot=True → Shape=(144, 12), Nulls=0, Zeros=0
  [3/23-3/28] impute=False, pivot=False → Shape=(720, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=True → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=True, pivot=False → Shape=(864, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=True → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=False, pivot=False → Shape=(864, 5), Nulls=0, Zeros=0


,dt,hr,ReserveZone,spp_res_zonal_wind_forecast_f,spp_res_zonal_wind_actual_a
0,2026-04-05,1,1,1587.70,1520.92
1,2026-04-05,1,2,888.76,589.00
2,2026-04-05,1,3,997.69,826.37
3,2026-04-05,1,4,3666.92,2727.00
4,2026-04-05,1,5,1803.36,1658.52


### 11. Wind Error

In [7]:
print("=== 11. Wind Error ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_we = wind.get_wind_error(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_we, period, impute)
# Extra: show_warnings
for s, e, period in PERIODS:
    df_sw = wind.get_wind_error(s, e, var_spec=["f", "a"], show_warnings=False)
    check_df(df_sw, period, True, extra_label="show_warnings=False")
display(df_we.head())


=== 11. Wind Error ===
  [3/23-3/28] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] [show_warnings=False] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] [show_warnings=False] impute=True → Shape=(144, 4), Nulls=0, Zeros=0


,dt,hr,spp_wind_error_f,spp_wind_error_a
0,2026-04-05,1,275.882,1536.772496
1,2026-04-05,2,260.100,1159.135000
2,2026-04-05,3,250.062,723.830000
3,2026-04-05,4,242.101,1289.002500
4,2026-04-05,5,268.708,1432.441667


### 12. BAA Zonal Wind Error (E and W)

In [45]:
print("=== 12. BAA Zonal Wind Error ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_wbe = wind.get_baa_zonal_wind_error(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], impute=impute)
        check_df(df_wbe, period, impute)
        display(df_wbe[:20])
# Extra: show_warnings
for s, e, period in PERIODS:
    df_sw = wind.get_baa_zonal_wind_error(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], show_warnings=False)
    check_df(df_sw, period, True, extra_label="show_warnings=False")
display(df_wbe.head())


=== 12. BAA Zonal Wind Error ===
  [3/23-3/28] impute=True → Shape=(144, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_baa_zonal_wind_error_a,spp_baa_zonal_wind_error_f
0,2026-03-23,1,E,-1239.605820,1081.540
1,2026-03-23,2,E,-1130.110840,1041.340
2,2026-03-23,3,E,-587.282520,1013.120
3,2026-03-23,4,E,192.128320,1004.310
4,2026-03-23,5,E,571.042510,970.805
5,2026-03-23,6,E,526.239990,949.484
6,2026-03-23,7,E,630.238319,943.810
7,2026-03-23,8,E,-2.809167,949.484
8,2026-03-23,9,E,-688.097500,943.810
9,2026-03-23,10,E,-2029.540834,949.484


  [3/23-3/28] impute=False → Shape=(144, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_baa_zonal_wind_error_a,spp_baa_zonal_wind_error_f
0,2026-03-23,1,E,-1239.605820,1081.540
1,2026-03-23,2,E,-1130.110840,1041.340
2,2026-03-23,3,E,-587.282520,1013.120
3,2026-03-23,4,E,192.128320,1004.310
4,2026-03-23,5,E,571.042510,970.805
5,2026-03-23,6,E,526.239990,949.484
6,2026-03-23,7,E,630.238319,943.810
7,2026-03-23,8,E,-2.809167,949.484
8,2026-03-23,9,E,-688.097500,943.810
9,2026-03-23,10,E,-2029.540834,949.484


  [4/5-4/10] impute=True → Shape=(288, 5), Nulls=0, Zeros=0


...

  [4/5-4/10] impute=False → Shape=(288, 5), Nulls=144, Zeros=0


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1610: UserWarning: No model-predicted BAA-zonal wind error forecast available for zone(s) ['W']; forecast will be NaN.
  warnings.warn(


,dt,hr,baa_zone,spp_baa_zonal_wind_error_a,spp_baa_zonal_wind_error_f
0,2026-04-05,1,E,1520.082496,275.882
1,2026-04-05,1,W,16.690000,NaN
2,2026-04-05,2,E,1104.855000,260.100
3,2026-04-05,2,W,54.280000,NaN
4,2026-04-05,3,E,636.960000,250.062
5,2026-04-05,3,W,86.870000,NaN
6,2026-04-05,4,E,1237.072500,242.101
7,2026-04-05,4,W,51.930000,NaN
8,2026-04-05,5,E,1401.661667,268.708
9,2026-04-05,5,W,30.780000,NaN


  [3/23-3/28] [show_warnings=False] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] [show_warnings=False] impute=True → Shape=(288, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_baa_zonal_wind_error_a,spp_baa_zonal_wind_error_f
0,2026-04-05,1,E,1520.082496,275.882
1,2026-04-05,1,W,16.690000,NaN
2,2026-04-05,2,E,1104.855000,260.100
3,2026-04-05,2,W,54.280000,NaN
4,2026-04-05,3,E,636.960000,250.062


### 13. VER Curtailment

In [9]:
print("=== 13. VER Curtailment ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_ver = wind.get_ver_curtailment(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_ver, period, impute)
display(df_ver.head())


=== 13. VER Curtailment ===
  [3/23-3/28] impute=True → Shape=(144, 4), Nulls=0, Zeros=3
  [3/23-3/28] impute=False → Shape=(144, 4), Nulls=0, Zeros=3
  [4/5-4/10] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 4), Nulls=0, Zeros=0


,dt,hr,spp_ver_curtailments_forecast_f,spp_ver_curtailments_actual_a
0,2026-04-05,1,201.044,68.847504
1,2026-04-05,2,201.044,5.765000
2,2026-04-05,3,201.044,13.270000
3,2026-04-05,4,189.201,11.097500
4,2026-04-05,5,186.902,4.408333


### 14. BAA Zonal VER Curtailment (E and W)

In [44]:
print("=== 14. BAA Zonal VER Curtailment ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_vb = wind.get_baa_zonal_ver_curtailment(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], pivot=pivot, impute=impute)
            print(df_vb[df_vb['dt']=='2026-04-06'])
            check_df(df_vb, period, impute, pivot)
# Extra: show_warnings
for s, e, period in PERIODS:
    df_sw = wind.get_baa_zonal_ver_curtailment(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], show_warnings=False)
    check_df(df_sw, period, True, extra_label="show_warnings=False")
display(df_vb.head())


=== 14. BAA Zonal VER Curtailment ===
Empty DataFrame
Columns: [dt, hr, e_spp_baa_zonal_ver_curtailments_actual_a, e_spp_baa_zonal_ver_curtailments_forecast_f]
Index: []
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 4), Nulls=0, Zeros=3
Empty DataFrame
Columns: [dt, hr, baa_zone, spp_baa_zonal_ver_curtailments_forecast_f, spp_baa_zonal_ver_curtailments_actual_a]
Index: []
  [3/23-3/28] impute=True, pivot=False → Shape=(144, 5), Nulls=0, Zeros=3


/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; forecast will be NaN.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; forecast will be NaN.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtail

Empty DataFrame
Columns: [dt, hr, e_spp_baa_zonal_ver_curtailments_actual_a, e_spp_baa_zonal_ver_curtailments_forecast_f]
Index: []
  [3/23-3/28] impute=False, pivot=True → Shape=(144, 4), Nulls=0, Zeros=3
Empty DataFrame
Columns: [dt, hr, baa_zone, spp_baa_zonal_ver_curtailments_forecast_f, spp_baa_zonal_ver_curtailments_actual_a]
Index: []
  [3/23-3/28] impute=False, pivot=False → Shape=(144, 5), Nulls=0, Zeros=3
            dt  hr  e_spp_baa_zonal_ver_curtailments_actual_a  \
24  2026-04-06   1                                1427.420776   
25  2026-04-06   2                                1177.218384   
26  2026-04-06   3                                 592.900818   
27  2026-04-06   4                                 500.237488   
28  2026-04-06   5                                 384.319153   
29  2026-04-06   6                                 324.763336   
30  2026-04-06   7                                 355.359161   
31  2026-04-06   8                                 352.160839

/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; using actual curtailment as forecast.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; forecast will be NaN.
  warnings.warn(
/var/www/python/Qingcheng/nighthawk/nighthawk/data/pipeline/common_functions/wind.py:1816: UserWarning: No BAA-zonal VER curtailment forecast available for zone(s) ['W']; forecast will be NaN.
  warnings.warn(


            dt  hr baa_zone  spp_baa_zonal_ver_curtailments_forecast_f  \
48  2026-04-06   1        E                                1180.450000   
49  2026-04-06   1        W                                   0.000000   
50  2026-04-06   2        E                                1193.510000   
51  2026-04-06   2        W                                   0.000000   
52  2026-04-06   3        E                                1012.160000   
53  2026-04-06   3        W                                   0.000000   
54  2026-04-06   4        E                                 862.020000   
55  2026-04-06   4        W                                   0.000000   
56  2026-04-06   5        E                                 871.582000   
57  2026-04-06   5        W                                   0.000000   
58  2026-04-06   6        E                                 841.974000   
59  2026-04-06   6        W                                   0.000000   
60  2026-04-06   7        E           

,dt,hr,baa_zone,spp_baa_zonal_ver_curtailments_forecast_f,spp_baa_zonal_ver_curtailments_actual_a
0,2026-04-05,1,E,201.044,68.847504
1,2026-04-05,1,W,NaN,0.000000
2,2026-04-05,2,E,201.044,5.765000
3,2026-04-05,2,W,NaN,0.000000
4,2026-04-05,3,E,201.044,13.270000


---
## SOLAR

### 15. Total Solar

In [11]:
print("=== 15. Total Solar ===")
# Note: get_total_solar forecast_type only applies to ERCOT (STPPF/COP_HSL/PVGRPP)
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_st = solar.get_total_solar(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_st, period, impute)
# Extra: forecast_type (ERCOT-only; testing default vs explicit for completeness)
for s, e, period in PERIODS:
    df_ft = solar.get_total_solar(s, e, var_spec=["f", "a"], forecast_type="STPPF")
    check_df(df_ft, period, False, extra_label="forecast_type=STPPF")
display(df_st.head())


=== 15. Total Solar ===
  [3/23-3/28] impute=True → Shape=(144, 4), Nulls=0, Zeros=70
  [3/23-3/28] impute=False → Shape=(144, 4), Nulls=0, Zeros=94
  [4/5-4/10] impute=True → Shape=(144, 4), Nulls=0, Zeros=113
  [4/5-4/10] impute=False → Shape=(144, 4), Nulls=0, Zeros=113
  [3/23-3/28] [forecast_type=STPPF] impute=False → Shape=(144, 4), Nulls=0, Zeros=94
  [4/5-4/10] [forecast_type=STPPF] impute=False → Shape=(144, 4), Nulls=0, Zeros=113


,dt,hr,spp_solar_total_forecast_f,spp_solar_total_actual_a
0,2026-04-05,1,0.0,0.0
1,2026-04-05,2,0.0,0.0
2,2026-04-05,3,0.0,0.0
3,2026-04-05,4,0.0,0.0
4,2026-04-05,5,0.0,0.0


### 16. BAA Zonal Solar (E and W)

In [12]:
print("=== 16. BAA Zonal Solar ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_sb = solar.get_baa_zonal_solar(s, e, baa_zones=BAA_ZONES, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_sb, period, impute, pivot)
display(df_sb.head())


=== 16. BAA Zonal Solar ===
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 4), Nulls=0, Zeros=70
  [3/23-3/28] impute=True, pivot=False → Shape=(144, 5), Nulls=0, Zeros=70
  [3/23-3/28] impute=False, pivot=True → Shape=(144, 4), Nulls=0, Zeros=94
  [3/23-3/28] impute=False, pivot=False → Shape=(144, 5), Nulls=0, Zeros=94
  [4/5-4/10] impute=True, pivot=True → Shape=(144, 6), Nulls=0, Zeros=233
  [4/5-4/10] impute=True, pivot=False → Shape=(288, 5), Nulls=0, Zeros=233
  [4/5-4/10] impute=False, pivot=True → Shape=(144, 6), Nulls=0, Zeros=233
  [4/5-4/10] impute=False, pivot=False → Shape=(288, 5), Nulls=0, Zeros=233


,dt,hr,baa_zone,spp_baa_zonal_solar_forecast_f,spp_baa_zonal_solar_actual_a
0,2026-04-05,1,E,0.0,0.0
1,2026-04-05,1,W,0.0,0.0
2,2026-04-05,2,E,0.0,0.0
3,2026-04-05,2,W,0.0,0.0
4,2026-04-05,3,E,0.0,0.0


### 17. Reserve Zonal Solar

In [13]:
print("=== 17. Reserve Zonal Solar ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        for pivot in [True, False]:
            df_sr = solar.get_res_zonal_solar(s, e, var_spec=["f", "a"], pivot=pivot, impute=impute)
            check_df(df_sr, period, impute, pivot)
display(df_sr.head())


=== 17. Reserve Zonal Solar ===
  [3/23-3/28] impute=True, pivot=True → Shape=(144, 12), Nulls=0, Zeros=570
  [3/23-3/28] impute=True, pivot=False → Shape=(720, 5), Nulls=0, Zeros=570
  [3/23-3/28] impute=False, pivot=True → Shape=(144, 12), Nulls=0, Zeros=570
  [3/23-3/28] impute=False, pivot=False → Shape=(720, 5), Nulls=0, Zeros=570
  [4/5-4/10] impute=True, pivot=True → Shape=(144, 14), Nulls=0, Zeros=610
  [4/5-4/10] impute=True, pivot=False → Shape=(864, 5), Nulls=0, Zeros=610
  [4/5-4/10] impute=False, pivot=True → Shape=(144, 14), Nulls=0, Zeros=610
  [4/5-4/10] impute=False, pivot=False → Shape=(864, 5), Nulls=0, Zeros=610


,dt,hr,ReserveZone,spp_res_zonal_solar_forecast_f,spp_res_zonal_solar_actual_a
0,2026-04-05,1,1,0.0,0.0
1,2026-04-05,2,1,0.0,0.0
2,2026-04-05,3,1,0.0,0.0
3,2026-04-05,4,1,0.0,0.0
4,2026-04-05,5,1,0.0,0.0


---
## GEN OUTAGE

### 18. Gen Outage — Total SPP

In [14]:
print("=== 18. Gen Outage by Level ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_go = genout.get_genoutage_by_level(s, e, area_list=["SPP"], var_spec=["f", "a"], impute=impute)
        check_df(df_go, period, impute)
# Extra: outage_type — for SPP, area_list drives granularity; outage_type is PJM/ERCOT-specific
# Testing BAA-zonal areas alongside total
for s, e, period in PERIODS:
    df_baa_go = genout.get_genoutage_by_level(s, e, area_list=["E", "W"], var_spec=["f", "a"])
    check_df(df_baa_go, period, True, extra_label="area_list=[E,W]")
display(df_go.head())


=== 18. Gen Outage by Level ===
  [3/23-3/28] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 5), Nulls=24, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 5), Nulls=0, Zeros=0
  [3/23-3/28] [area_list=[E,W]] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] [area_list=[E,W]] impute=True → Shape=(288, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_genoutage_forecast_f,spp_genoutage_actual_a
0,2026-04-05,1,SPP,27225.7,27229.5
1,2026-04-05,2,SPP,27221.7,27219.2
2,2026-04-05,3,SPP,27222.3,27219.2
3,2026-04-05,4,SPP,27222.3,27219.2
4,2026-04-05,5,SPP,27219.2,26609.8


### 19. Gen Outage — BAA Zonal (E and W)

In [15]:
print("=== 19. Gen Outage BAA Zonal + FuelType ===")
# get_genoutage_by_level (E/W areas)
for s, e, period in PERIODS:
    for impute in [True, False]:
        df = genout.get_genoutage_by_level(s, e, area_list=["E", "W"], var_spec=["f", "a"], impute=impute)
        check_df(df, period, impute, extra_label="area=[E,W]")

# get_genoutage_fuel_type — impute + intp_method (only "linear" supported)
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_gft = fueltype.get_genoutage_fuel_type(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_gft, period, impute, extra_label="genoutage_fuel_type")
display(df_gft.head())


Objects created for SPP  |  2026-04-01 → 2026-04-01
(48, 5)


,dt,hr,baa_zone,spp_genoutage_forecast_f,spp_genoutage_actual_a
0,2026-04-07,1,E,26588.4,27190.4
1,2026-04-07,1,W,1739.1,1858.0
2,2026-04-07,2,E,26790.4,27392.4
3,2026-04-07,2,W,1739.1,1858.0
4,2026-04-07,3,E,26790.4,27392.4
5,2026-04-07,3,W,1858.0,1858.0
6,2026-04-07,4,E,26790.4,27392.4
7,2026-04-07,4,W,1858.0,1858.0
8,2026-04-07,5,E,27392.4,26613.2
9,2026-04-07,5,W,1858.0,1939.6


(24, 14)


,dt,hr,spp_fuelmixoutage_coal_forecast_f,spp_fuelmixoutage_dieselfueloil_forecast_f,spp_fuelmixoutage_hydro_forecast_f,spp_fuelmixoutage_naturalgas_forecast_f,spp_fuelmixoutage_solar_forecast_f,spp_fuelmixoutage_wind_forecast_f,spp_fuelmixoutage_coal_actual_a,spp_fuelmixoutage_dieselfueloil_actual_a,spp_fuelmixoutage_hydro_actual_a,spp_fuelmixoutage_naturalgas_actual_a,spp_fuelmixoutage_solar_actual_a,spp_fuelmixoutage_wind_actual_a
0,2026-04-07,1,9912.8,119.8,1656.1,14793.8,1247.5,283.0,10736.7,119.8,1656.1,14688.8,1247.5,283.0
1,2026-04-07,2,9912.8,119.8,1656.1,14995.8,1247.5,283.0,10736.7,119.8,1656.1,14890.8,1247.5,283.0
2,2026-04-07,3,10031.7,119.8,1656.1,14995.8,1247.5,283.0,10736.7,119.8,1656.1,14890.8,1247.5,283.0
3,2026-04-07,4,10031.7,119.8,1656.1,14995.8,1247.5,283.0,10736.7,119.8,1656.1,14890.8,1247.5,283.0
4,2026-04-07,5,10736.7,119.8,1656.1,14890.8,1247.5,283.0,9990.8,119.8,1656.1,14925.4,1255.0,283.0
5,2026-04-07,6,10736.7,119.8,1656.1,15031.8,1247.5,283.0,10265.8,119.8,1656.1,15066.4,1255.0,283.0
6,2026-04-07,7,10736.7,119.8,1676.1,15031.8,1247.5,283.0,10265.8,119.8,1676.1,15066.4,1255.0,283.0
7,2026-04-07,8,10736.7,119.8,1676.1,15377.8,1097.5,283.0,10265.8,119.8,1676.1,15870.4,1105.0,283.0
8,2026-04-07,9,10736.7,119.8,1676.1,15377.8,1097.5,878.0,10265.8,119.8,1688.1,15870.4,1105.0,878.0
9,2026-04-07,10,10736.7,119.8,1676.1,15377.8,1097.5,878.0,10265.8,119.8,1688.1,15870.4,1105.0,878.0


---
## GEN MIX (FUEL TYPE)

### 20. All 5 GenMix Nighthawk Functions — 2026-04-01

| # | Function | Table | Fuel types | Zones | a/f |
|---|---|---|---|---|---|
| 1 | `get_gen_fuel_type` | `GenMixHourly` | Hydro, Nuclear, NaturalGas | SPP total | both |
| 2 | `get_gen_fuel_actual` | `GenMixHourly` | all 10 | SPP total | actual only |
| 3 | `get_baa_zonal_gen_fuel_type` | `BAAZonalGenMixHourly` | Hydro, Nuclear, NaturalGas | E + W | both |
| 4 | `get_baa_zonal_gen_fuel_actual` | `BAAZonalGenMixHourly` | all 10 | E + W | actual only |
| 5 | `get_spp_gas_output` *(deprecated)* | `GenMixHourly` | NaturalGas only | SPP total | both |

In [42]:
from nighthawk.data.pipeline.var_handler.fuel_type import FuelType
from nighthawk.data.pipeline.var_handler.spp_specific.gas_price_effect import get_spp_gas_output
ft = FuelType("SPP")
print("=== 20. All 5 GenMix Functions ===")

# # 1. get_gen_fuel_type — impute T/F; intp_method only supports "linear"
# for s, e, period in PERIODS:
#     for impute in [True, False]:
#         df1 = ft.get_gen_fuel_type(s, e, var_spec=["a", "f"], impute=impute)
#         check_df(df1, period, impute, extra_label="gen_fuel_type")

# # 2. get_gen_fuel_actual — pivot T/F, impute T/F, show_pct_flag, nday_pctl_flag, nday
# for s, e, period in PERIODS:
#     for impute in [True, False]:
#         for pivot in [True, False]:
#             df2 = ft.get_gen_fuel_actual(s, e, impute=impute, pivot=pivot)
#             check_df(df2, period, impute, pivot, extra_label="gen_fuel_actual")
# # Extra: show_pct_flag
# for s, e, period in PERIODS:
#     df_pct = ft.get_gen_fuel_actual(s, e, show_pct_flag=True)
#     check_df(df_pct, period, False, extra_label="gen_fuel_actual show_pct_flag=True")
# # Extra: nday_pctl_flag + nday
# for s, e, period in PERIODS:
#     df_nd = ft.get_gen_fuel_actual(s, e, nday_pctl_flag=True, nday=[7, 30])
#     check_df(df_nd, period, False, extra_label="gen_fuel_actual nday=[7,30]")

# # 3. get_baa_zonal_gen_fuel_type — pivot T/F, impute T/F
# for s, e, period in PERIODS:
#     for impute in [True, False]:
#         for pivot in [True, False]:
#             df3 = ft.get_baa_zonal_gen_fuel_type(s, e, baa_zones=["E", "W"], var_spec=["a", "f"], pivot=pivot, impute=impute)
#             check_df(df3, period, impute, pivot, extra_label="baa_gen_fuel_type")

# # 4. get_baa_zonal_gen_fuel_actual — pivot T/F, impute T/F, show_pct_flag, nday_pctl_flag
# for s, e, period in PERIODS:
#     for impute in [True, False]:
#         for pivot in [True, False]:
#             df4 = ft.get_baa_zonal_gen_fuel_actual(s, e, baa_zones=["E", "W"], pivot=pivot, impute=impute)
#             check_df(df4, period, impute, pivot, extra_label="baa_gen_fuel_actual")
# # Extra: show_pct_flag
# for s, e, period in PERIODS:
#     df_pct4 = ft.get_baa_zonal_gen_fuel_actual(s, e, baa_zones=["E", "W"], show_pct_flag=True)
#     check_df(df_pct4, period, False, extra_label="baa_gen_fuel_actual show_pct_flag=True")

# 5. get_spp_gas_output (deprecated) — impute T/F
for s, e, period in PERIODS:
    for impute in [True, False]:
        df5 = get_spp_gas_output(s, e, var_spec=["a", "f"], impute=impute)
        check_df(df5, period, impute, extra_label="spp_gas_output")
display(df5.head())


=== 20. All 5 GenMix Functions ===
  [3/23-3/28] [spp_gas_output] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [3/23-3/28] [spp_gas_output] impute=False → Shape=(144, 4), Nulls=28, Zeros=0


/tmp/ipykernel_3255507/3365170271.py:48: DeprecationWarning: Call to deprecated function get_spp_gas_output.
  df5 = get_spp_gas_output(s, e, var_spec=["a", "f"], impute=impute)
/tmp/ipykernel_3255507/3365170271.py:48: DeprecationWarning: Call to deprecated function get_spp_gas_output.
  df5 = get_spp_gas_output(s, e, var_spec=["a", "f"], impute=impute)
/tmp/ipykernel_3255507/3365170271.py:48: DeprecationWarning: Call to deprecated function get_spp_gas_output.
  df5 = get_spp_gas_output(s, e, var_spec=["a", "f"], impute=impute)


  [4/5-4/10] [spp_gas_output] impute=True → Shape=(144, 4), Nulls=0, Zeros=0
  [4/5-4/10] [spp_gas_output] impute=False → Shape=(144, 4), Nulls=24, Zeros=0


/tmp/ipykernel_3255507/3365170271.py:48: DeprecationWarning: Call to deprecated function get_spp_gas_output.
  df5 = get_spp_gas_output(s, e, var_spec=["a", "f"], impute=impute)


,dt,hr,spp_fuelmix_naturalgas_actual_a,spp_fuelmix_naturalgas_forecast_f
0,2026-04-05,1,11608.5,NaN
1,2026-04-05,2,10784.6,NaN
2,2026-04-05,3,10721.2,NaN
3,2026-04-05,4,11288.9,NaN
4,2026-04-05,5,11582.5,NaN


### 21. Gen Outage by Fuel Type

In [20]:
print("=== 21. Gen Outage Total ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_genout_total = genout.get_genoutage_by_level(s, e, area_list=["SPP"], var_spec=["f", "a"], impute=impute)
        check_df(df_genout_total, period, impute)
display(df_genout_total.head())


=== 21. Gen Outage Total ===
  [3/23-3/28] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 5), Nulls=24, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 5), Nulls=0, Zeros=0


,dt,hr,baa_zone,spp_genoutage_forecast_f,spp_genoutage_actual_a
0,2026-04-05,1,SPP,27225.7,27229.5
1,2026-04-05,2,SPP,27221.7,27219.2
2,2026-04-05,3,SPP,27222.3,27219.2
3,2026-04-05,4,SPP,27222.3,27219.2
4,2026-04-05,5,SPP,27219.2,26609.8


In [23]:
print("=== 21b. get_genoutage_fuel_type ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_outage_fuel = fueltype.get_genoutage_fuel_type(s, e, var_spec=["f", "a"], impute=impute)
        check_df(df_outage_fuel, period, impute)
df_outage_fuel['total_f'] = df_outage_fuel.filter(regex='_f$').sum(axis=1)
df_outage_fuel['total_a'] = df_outage_fuel.filter(regex='_a$').sum(axis=1)
display(df_outage_fuel.head())


=== 21b. get_genoutage_fuel_type ===
  [3/23-3/28] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(212, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 14), Nulls=0, Zeros=0


,dt,hr,spp_fuelmixoutage_coal_forecast_f,spp_fuelmixoutage_dieselfueloil_forecast_f,spp_fuelmixoutage_hydro_forecast_f,spp_fuelmixoutage_naturalgas_forecast_f,spp_fuelmixoutage_solar_forecast_f,spp_fuelmixoutage_wind_forecast_f,spp_fuelmixoutage_coal_actual_a,spp_fuelmixoutage_dieselfueloil_actual_a,spp_fuelmixoutage_hydro_actual_a,spp_fuelmixoutage_naturalgas_actual_a,spp_fuelmixoutage_solar_actual_a,spp_fuelmixoutage_wind_actual_a,total_f,total_a
0,2026-04-05,1,8995.8,75.8,1615.3,14992.3,970.5,263.0,9532.8,75.8,1620.3,14442.3,970.5,263.0,26912.7,26904.7
1,2026-04-05,2,8995.8,75.8,1615.3,14992.3,970.5,263.0,9532.8,75.8,1620.3,14442.3,970.5,263.0,26912.7,26904.7
2,2026-04-05,3,8995.8,75.8,1615.3,14992.3,970.5,263.0,9532.8,75.8,1620.3,14442.3,970.5,263.0,26912.7,26904.7
3,2026-04-05,4,8995.8,75.8,1615.3,14992.3,970.5,263.0,9532.8,75.8,1620.3,14442.3,970.5,263.0,26912.7,26904.7
4,2026-04-05,5,9532.8,75.8,1620.3,14442.3,970.5,263.0,9532.8,75.8,1620.3,13830.3,970.5,263.0,26904.7,26292.7


In [24]:
# ── DIAGNOSTIC: why fuel-type sum != GeneratorOutageNew total ──────────────
import pandas as pd
from nighthawk.util import connections, sql_functions

# 1. Raw GeneratorOutageNew — dt/hr already in SPP market time
df_gon = sql_functions.download_df_from_sql_db(
    "SELECT dt, hr, ForecastOutage, ActualOutage "
    "FROM spp_physical.GeneratorOutageNew "
    "WHERE dt = '2026-04-01' ORDER BY hr"
)
df_gon['dt'] = df_gon['dt'].astype(str)
print('GeneratorOutageNew columns:', df_gon.columns.tolist())
print('Rows:', len(df_gon))
display(df_gon)

# 2. Raw fuel-type tables — dthr is UTC hour-ending
#    nighthawk converts: tz_localize(UTC) -> tz_convert(US/Central) - 1h, then hr = local_hour + 1
fuel_cols = ['CoalMW','DieselFuelOilMW','HydroMW','NaturalGasMW','SolarMW','WindMW']

df_ftype_f = sql_functions.download_df_from_sql_db(
    "SELECT dthr, CoalMW, DieselFuelOilMW, HydroMW, NaturalGasMW, SolarMW, WindMW "
    "FROM spp_physical.GenOutagebyFuelTypeForecast "
    "WHERE DATE(dthr) >= DATE_SUB('2026-04-01', INTERVAL 1 DAY) "
    "  AND DATE(dthr) <= DATE_ADD('2026-04-01', INTERVAL 1 DAY) ORDER BY dthr"
)
df_ftype_a = sql_functions.download_df_from_sql_db(
    "SELECT dthr, CoalMW, DieselFuelOilMW, HydroMW, NaturalGasMW, SolarMW, WindMW "
    "FROM spp_physical.GenOutagebyFuelTypeActual "
    "WHERE DATE(dthr) >= DATE_SUB('2026-04-01', INTERVAL 1 DAY) "
    "  AND DATE(dthr) <= DATE_ADD('2026-04-01', INTERVAL 1 DAY) ORDER BY dthr"
)

for df, tag in [(df_ftype_f, 'Forecast'), (df_ftype_a, 'Actual')]:
    df['dthr'] = pd.to_datetime(df['dthr']).dt.tz_localize('UTC').dt.tz_convert('US/Central') - pd.Timedelta('1h')
    df['dt']   = df['dthr'].dt.strftime('%Y-%m-%d')
    df['hr']   = df['dthr'].dt.hour + 1
    df.drop(columns=['dthr'], inplace=True)
    df['fuel_sum'] = df[fuel_cols].sum(axis=1)
    print(f'GenOutagebyFuelType{tag} rows after tz-convert:', len(df))
    display(df[df['dt'] == '2026-04-01'][['dt','hr','fuel_sum'] + fuel_cols])

# 3. Side-by-side on target date
df_f_day = df_ftype_f[df_ftype_f['dt'] == '2026-04-01'][['dt','hr','fuel_sum']].rename(columns={'fuel_sum':'fuel_sum_f'})
df_a_day = df_ftype_a[df_ftype_a['dt'] == '2026-04-01'][['dt','hr','fuel_sum']].rename(columns={'fuel_sum':'fuel_sum_a'})
df_gon_day = df_gon[df_gon['dt'] == '2026-04-01'][['dt','hr','ForecastOutage','ActualOutage']]

cmp = df_gon_day.merge(df_f_day, on=['dt','hr'], how='outer').merge(df_a_day, on=['dt','hr'], how='outer')
cmp['diff_f'] = cmp['ForecastOutage'] - cmp['fuel_sum_f']
cmp['diff_a'] = cmp['ActualOutage']   - cmp['fuel_sum_a']
print('\n=== Comparison: GeneratorOutageNew vs fuel_type_sum ===')
display(cmp)
print('Max |diff_f|:', cmp['diff_f'].abs().max(), '  Max |diff_a|:', cmp['diff_a'].abs().max())
print('Hours with diff_f != 0:', (cmp['diff_f'].abs() > 0.01).sum())
print('Hours with diff_a != 0:', (cmp['diff_a'].abs() > 0.01).sum())

# 4. Inspect all columns of GeneratorOutageNew (may reveal extra fuel type fields)
df_raw = sql_functions.download_df_from_sql_db("SELECT * FROM spp_physical.GeneratorOutageNew WHERE dt = '2026-04-01' LIMIT 3")
print('\nAll columns in GeneratorOutageNew:', df_raw.columns.tolist())
display(df_raw)


GeneratorOutageNew columns: ['dt', 'hr', 'ForecastOutage', 'ActualOutage']
Rows: 24


,dt,hr,ForecastOutage,ActualOutage
0,2026-04-01,1,25715.9,25197.9
1,2026-04-01,2,25728.3,25198.1
2,2026-04-01,3,25878.9,25348.7
3,2026-04-01,4,25878.9,25348.7
4,2026-04-01,5,25712.9,25182.7
5,2026-04-01,6,25612.9,25182.7
6,2026-04-01,7,25897.1,25171.9
7,2026-04-01,8,25907.1,25179.9
8,2026-04-01,9,25907.1,25180.5
9,2026-04-01,10,25907.7,25183.5


GenOutagebyFuelTypeForecast rows after tz-convert: 72


,dt,hr,fuel_sum,CoalMW,DieselFuelOilMW,HydroMW,NaturalGasMW,SolarMW,WindMW
30,2026-04-01,1,25603.5,8488.8,72.0,936.0,14325.5,1155.7,625.5
31,2026-04-01,2,25613.5,8488.8,72.0,936.0,14335.5,1155.7,625.5
32,2026-04-01,3,25613.5,8488.8,72.0,936.0,14335.5,1155.7,625.5
33,2026-04-01,4,25613.5,8488.8,72.0,936.0,14335.5,1155.7,625.5
34,2026-04-01,5,24922.3,8488.8,72.0,890.0,13762.5,1003.5,705.5
35,2026-04-01,6,24922.3,8488.8,72.0,890.0,13762.5,1003.5,705.5
36,2026-04-01,7,25007.3,8488.8,72.0,890.0,13847.5,1003.5,705.5
37,2026-04-01,8,24902.3,8488.8,72.0,890.0,13892.5,853.5,705.5
38,2026-04-01,9,25024.3,8488.8,266.0,890.0,13892.5,781.5,705.5
39,2026-04-01,10,25024.3,8488.8,266.0,890.0,13892.5,781.5,705.5


GenOutagebyFuelTypeActual rows after tz-convert: 106


,dt,hr,fuel_sum,CoalMW,DieselFuelOilMW,HydroMW,NaturalGasMW,SolarMW,WindMW
60,2026-04-01,1,25340.6,8488.8,72.0,825.0,13897.6,1003.5,1053.7
61,2026-04-01,1,24912.3,8488.8,72.0,890.0,13752.5,1003.5,705.5
62,2026-04-01,2,24922.3,8488.8,72.0,890.0,13762.5,1003.5,705.5
63,2026-04-01,2,25350.6,8488.8,72.0,825.0,13907.6,1003.5,1053.7
64,2026-04-01,3,24922.3,8488.8,72.0,890.0,13762.5,1003.5,705.5
65,2026-04-01,3,25350.6,8488.8,72.0,825.0,13907.6,1003.5,1053.7
66,2026-04-01,4,25350.6,8488.8,72.0,825.0,13907.6,1003.5,1053.7
67,2026-04-01,4,24922.3,8488.8,72.0,890.0,13762.5,1003.5,705.5
68,2026-04-01,5,25480.7,8938.8,72.0,890.0,13973.9,781.5,824.5
69,2026-04-01,6,25480.7,8938.8,72.0,890.0,13973.9,781.5,824.5



=== Comparison: GeneratorOutageNew vs fuel_type_sum ===


,dt,hr,ForecastOutage,ActualOutage,fuel_sum_f,fuel_sum_a,diff_f,diff_a
0,2026-04-01,1,25715.9,25197.9,25603.5,25340.6,112.4,-142.7
1,2026-04-01,1,25715.9,25197.9,25603.5,24912.3,112.4,285.6
2,2026-04-01,2,25728.3,25198.1,25613.5,24922.3,114.8,275.8
3,2026-04-01,2,25728.3,25198.1,25613.5,25350.6,114.8,-152.5
4,2026-04-01,3,25878.9,25348.7,25613.5,24922.3,265.4,426.4
5,2026-04-01,3,25878.9,25348.7,25613.5,25350.6,265.4,-1.9
6,2026-04-01,4,25878.9,25348.7,25613.5,25350.6,265.4,-1.9
7,2026-04-01,4,25878.9,25348.7,25613.5,24922.3,265.4,426.4
8,2026-04-01,5,25712.9,25182.7,24922.3,25480.7,790.6,-298.0
9,2026-04-01,6,25612.9,25182.7,24922.3,25480.7,690.6,-298.0


Max |diff_f|: 1004.7999999999993   Max |diff_a|: 782.7000000000044
Hours with diff_f != 0: 28
Hours with diff_a != 0: 28

All columns in GeneratorOutageNew: ['dt', 'hr', 'ActualOutage', 'ForecastOutage']


,dt,hr,ActualOutage,ForecastOutage
0,2026-04-01,1,25197.9,25715.9
1,2026-04-01,2,25198.1,25728.3
2,2026-04-01,3,25348.7,25878.9


### 21b. get_genoutage_fuel_type — market-total outage by fuel type

SPP only. Reads `spp_physical.GenOutagebyFuelTypeForecast` and `GenOutagebyFuelTypeActual`.

In [26]:
print("=== 21b. get_genoutage_fuel_type (impute=True check) ===")
for s, e, period in PERIODS:
    df_outage_fuel = fueltype.get_genoutage_fuel_type(s, e, var_spec=["f", "a"], impute=True)
    check_df(df_outage_fuel, period, True)
print(df_outage_fuel)


=== 21b. get_genoutage_fuel_type (impute=True check) ===
  [3/23-3/28] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
                dt  hr  spp_fuelmixoutage_coal_forecast_f  \
0  0    2026-04-05   1                             8995.8   
   24   2026-04-06   1                             9685.8   
   48   2026-04-07   1                             9912.8   
   72   2026-04-08   1                            10810.7   
   96   2026-04-09   1                            10010.8   
...            ...  ..                                ...   
23 47   2026-04-06  24                            10105.7   
   71   2026-04-07  24                            10736.7   
   95   2026-04-08  24                             9560.8   
   119  2026-04-09  24                             9960.8   
   143  2026-04-10  24                             9290.8   

        spp_fuelmixoutage_dieselfueloil_forecast_f  \
0  0                             

### 21c. get_baa_zonal_genoutage_fuel_type — BAA-zonal outage by fuel type

SPP only. Reads `spp_physical.BAAZonalGenOutagebyFuelTypeForecast` and `BAAZonalGenOutagebyFuelTypeActual`.

In [27]:
print("=== 21c. get_baa_zonal_genoutage_fuel_type ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        df_baa_ofl = fueltype.get_baa_zonal_genoutage_fuel_type(s, e, baa_zones=["E", "W"], var_spec=["f", "a"], impute=impute)
        check_df(df_baa_ofl, period, impute)
print(df_baa_ofl.columns.tolist())
display(df_baa_ofl.head())


=== 21c. get_baa_zonal_genoutage_fuel_type ===
  [3/23-3/28] impute=True → Shape=(144, 15), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 15), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(288, 15), Nulls=0, Zeros=728
  [4/5-4/10] impute=False → Shape=(288, 15), Nulls=0, Zeros=728
['dt', 'hr', 'baa_zone', 'spp_baa_zonal_fuelmixoutage_coal_forecast_f', 'spp_baa_zonal_fuelmixoutage_dieselfueloil_forecast_f', 'spp_baa_zonal_fuelmixoutage_hydro_forecast_f', 'spp_baa_zonal_fuelmixoutage_naturalgas_forecast_f', 'spp_baa_zonal_fuelmixoutage_solar_forecast_f', 'spp_baa_zonal_fuelmixoutage_wind_forecast_f', 'spp_baa_zonal_fuelmixoutage_coal_actual_a', 'spp_baa_zonal_fuelmixoutage_dieselfueloil_actual_a', 'spp_baa_zonal_fuelmixoutage_hydro_actual_a', 'spp_baa_zonal_fuelmixoutage_naturalgas_actual_a', 'spp_baa_zonal_fuelmixoutage_solar_actual_a', 'spp_baa_zonal_fuelmixoutage_wind_actual_a']


,dt,hr,baa_zone,spp_baa_zonal_fuelmixoutage_coal_forecast_f,spp_baa_zonal_fuelmixoutage_dieselfueloil_forecast_f,spp_baa_zonal_fuelmixoutage_hydro_forecast_f,spp_baa_zonal_fuelmixoutage_naturalgas_forecast_f,spp_baa_zonal_fuelmixoutage_solar_forecast_f,spp_baa_zonal_fuelmixoutage_wind_forecast_f,spp_baa_zonal_fuelmixoutage_coal_actual_a,spp_baa_zonal_fuelmixoutage_dieselfueloil_actual_a,spp_baa_zonal_fuelmixoutage_hydro_actual_a,spp_baa_zonal_fuelmixoutage_naturalgas_actual_a,spp_baa_zonal_fuelmixoutage_solar_actual_a,spp_baa_zonal_fuelmixoutage_wind_actual_a
0,2026-04-05,1,E,8738.8,75.8,779.0,14965.3,970.5,263.0,9275.8,75.8,779.0,14415.3,970.5,263.0
1,2026-04-05,1,W,257.0,0.0,836.3,27.0,0.0,0.0,257.0,0.0,841.3,27.0,0.0,0.0
2,2026-04-05,2,E,8738.8,75.8,779.0,14965.3,970.5,263.0,9275.8,75.8,779.0,14415.3,970.5,263.0
3,2026-04-05,2,W,257.0,0.0,836.3,27.0,0.0,0.0,257.0,0.0,841.3,27.0,0.0,0.0
4,2026-04-05,3,E,8738.8,75.8,779.0,14965.3,970.5,263.0,9275.8,75.8,779.0,14415.3,970.5,263.0


### 21d. get_data_and_mapping_for_genoutage_fuel_type — wrapper with node mapping

Same as `get_genoutage_fuel_type` but returns `(data, mapping)` keyed by node.

In [28]:
from nighthawk.data.pipeline.var_handler.fuel_type import get_data_and_mapping_for_genoutage_fuel_type
print("=== 21d. get_data_and_mapping_for_genoutage_fuel_type ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        data, mapping = get_data_and_mapping_for_genoutage_fuel_type(
            node_list=[636], opexchange="SPP", start_dt=s, end_dt=e,
            fuel_type=["Coal","DieselFuelOil","Hydro","NaturalGas","Solar","Wind"], impute=impute)
        check_df(data, period, impute, extra_label="fuel_type=all6")
# Extra: fuel_type=["All"] vs explicit list
for s, e, period in PERIODS:
    data_all, _ = get_data_and_mapping_for_genoutage_fuel_type(
        node_list=[636], opexchange="SPP", start_dt=s, end_dt=e, fuel_type=["All"])
    check_df(data_all, period, False, extra_label="fuel_type=[All]")
display(data.head()); display(mapping)


=== 21d. get_data_and_mapping_for_genoutage_fuel_type ===
  [3/23-3/28] [fuel_type=all6] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
  [3/23-3/28] [fuel_type=all6] impute=False → Shape=(212, 14), Nulls=0, Zeros=0
  [4/5-4/10] [fuel_type=all6] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] [fuel_type=all6] impute=False → Shape=(144, 14), Nulls=0, Zeros=0
  [3/23-3/28] [fuel_type=[All]] impute=False → Shape=(212, 14), Nulls=0, Zeros=0
  [4/5-4/10] [fuel_type=[All]] impute=False → Shape=(144, 14), Nulls=0, Zeros=0


,dt,hr,spp_fuelmixoutage_coal_forecast_f,spp_fuelmixoutage_coal_actual_a,spp_fuelmixoutage_dieselfueloil_forecast_f,spp_fuelmixoutage_dieselfueloil_actual_a,spp_fuelmixoutage_hydro_forecast_f,spp_fuelmixoutage_hydro_actual_a,spp_fuelmixoutage_naturalgas_forecast_f,spp_fuelmixoutage_naturalgas_actual_a,spp_fuelmixoutage_solar_forecast_f,spp_fuelmixoutage_solar_actual_a,spp_fuelmixoutage_wind_forecast_f,spp_fuelmixoutage_wind_actual_a
0,2026-04-05,1,8995.8,9532.8,75.8,75.8,1615.3,1620.3,14992.3,14442.3,970.5,970.5,263.0,263.0
1,2026-04-05,2,8995.8,9532.8,75.8,75.8,1615.3,1620.3,14992.3,14442.3,970.5,970.5,263.0,263.0
2,2026-04-05,3,8995.8,9532.8,75.8,75.8,1615.3,1620.3,14992.3,14442.3,970.5,970.5,263.0,263.0
3,2026-04-05,4,8995.8,9532.8,75.8,75.8,1615.3,1620.3,14992.3,14442.3,970.5,970.5,263.0,263.0
4,2026-04-05,5,9532.8,9532.8,75.8,75.8,1620.3,1620.3,14442.3,13830.3,970.5,970.5,263.0,263.0


,node_num,var_name
0,636,spp_fuelmixoutage_coal_forecast_f
1,636,spp_fuelmixoutage_coal_actual_a
2,636,spp_fuelmixoutage_dieselfueloil_forecast_f
3,636,spp_fuelmixoutage_dieselfueloil_actual_a
4,636,spp_fuelmixoutage_hydro_forecast_f
5,636,spp_fuelmixoutage_hydro_actual_a
6,636,spp_fuelmixoutage_naturalgas_forecast_f
7,636,spp_fuelmixoutage_naturalgas_actual_a
8,636,spp_fuelmixoutage_solar_forecast_f
9,636,spp_fuelmixoutage_solar_actual_a


### 21e. get_data_and_mapping_for_baa_zonal_genoutage_fuel_type — BAA-zonal wrapper with node mapping

In [29]:
from nighthawk.data.pipeline.var_handler.fuel_type import get_data_and_mapping_for_baa_zonal_genoutage_fuel_type
print("=== 21e. get_data_and_mapping_for_baa_zonal_genoutage_fuel_type ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        data_b, mapping_b = get_data_and_mapping_for_baa_zonal_genoutage_fuel_type(
            node_list=[636], opexchange="SPP", start_dt=s, end_dt=e, fuel_type=["All"], impute=impute)
        check_df(data_b, period, impute)
# Extra: specific fuel types
for s, e, period in PERIODS:
    data_b2, _ = get_data_and_mapping_for_baa_zonal_genoutage_fuel_type(
        node_list=[636], opexchange="SPP", start_dt=s, end_dt=e,
        fuel_type=["Coal","NaturalGas","Wind"])
    check_df(data_b2, period, False, extra_label="fuel_type=[Coal,NaturalGas,Wind]")
display(data_b.head()); display(mapping_b)


=== 21e. get_data_and_mapping_for_baa_zonal_genoutage_fuel_type ===
  [3/23-3/28] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
  [3/23-3/28] impute=False → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=True → Shape=(144, 14), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(144, 14), Nulls=0, Zeros=0
  [3/23-3/28] [fuel_type=[Coal,NaturalGas,Wind]] impute=False → Shape=(144, 8), Nulls=0, Zeros=0
  [4/5-4/10] [fuel_type=[Coal,NaturalGas,Wind]] impute=False → Shape=(144, 8), Nulls=0, Zeros=0


,dt,hr,e_spp_baa_zonal_fuelmixoutage_coal_actual_a,e_spp_baa_zonal_fuelmixoutage_coal_forecast_f,e_spp_baa_zonal_fuelmixoutage_dieselfueloil_actual_a,e_spp_baa_zonal_fuelmixoutage_dieselfueloil_forecast_f,e_spp_baa_zonal_fuelmixoutage_hydro_actual_a,e_spp_baa_zonal_fuelmixoutage_hydro_forecast_f,e_spp_baa_zonal_fuelmixoutage_naturalgas_actual_a,e_spp_baa_zonal_fuelmixoutage_naturalgas_forecast_f,e_spp_baa_zonal_fuelmixoutage_solar_actual_a,e_spp_baa_zonal_fuelmixoutage_solar_forecast_f,e_spp_baa_zonal_fuelmixoutage_wind_actual_a,e_spp_baa_zonal_fuelmixoutage_wind_forecast_f
0,2026-04-05,1,9275.8,8738.8,75.8,75.8,779.0,779.0,14415.3,14965.3,970.5,970.5,263.0,263.0
1,2026-04-05,2,9275.8,8738.8,75.8,75.8,779.0,779.0,14415.3,14965.3,970.5,970.5,263.0,263.0
2,2026-04-05,3,9275.8,8738.8,75.8,75.8,779.0,779.0,14415.3,14965.3,970.5,970.5,263.0,263.0
3,2026-04-05,4,9275.8,8738.8,75.8,75.8,779.0,779.0,14415.3,14965.3,970.5,970.5,263.0,263.0
4,2026-04-05,5,9275.8,9275.8,75.8,75.8,779.0,779.0,13803.3,14415.3,970.5,970.5,263.0,263.0


,node_num,var_name
0,636,e_spp_baa_zonal_fuelmixoutage_coal_actual_a
1,636,e_spp_baa_zonal_fuelmixoutage_coal_forecast_f
2,636,e_spp_baa_zonal_fuelmixoutage_dieselfueloil_ac...
3,636,e_spp_baa_zonal_fuelmixoutage_dieselfueloil_fo...
4,636,e_spp_baa_zonal_fuelmixoutage_hydro_actual_a
5,636,e_spp_baa_zonal_fuelmixoutage_hydro_forecast_f
6,636,e_spp_baa_zonal_fuelmixoutage_naturalgas_actual_a
7,636,e_spp_baa_zonal_fuelmixoutage_naturalgas_forec...
8,636,e_spp_baa_zonal_fuelmixoutage_solar_actual_a
9,636,e_spp_baa_zonal_fuelmixoutage_solar_forecast_f


---
## GEN MIX RAW TABLES

### 22. GenMix 5-min (spp_physical.GenMix)

In [30]:
print("=== 22. GenMix 5-min ===")
for s, e, period in PERIODS:
    df_genmix = sql_functions.download_df_from_sql_db(
        f"SELECT IntervalEnd, GMTIntervalEnd, CoalTotal, WindTotal, SolarTotal, NaturalGasTotal, AverageActualLoad "
        f"FROM spp_physical.GenMix "
        f"WHERE DATE(IntervalEnd) BETWEEN '{s}' AND '{e}' ORDER BY IntervalEnd")
    check_df(df_genmix, period, False)
df_genmix.tail(5)


=== 22. GenMix 5-min ===
  [3/23-3/28] impute=False → Shape=(1691, 7), Nulls=0, Zeros=47
  [4/5-4/10] impute=False → Shape=(1728, 7), Nulls=0, Zeros=19


,IntervalEnd,GMTIntervalEnd,CoalTotal,WindTotal,SolarTotal,NaturalGasTotal,AverageActualLoad
1723,2026-04-10 23:35:00,2026-04-11 04:35:00,6032.5,17634.5,0.2,5435.6,30663.3
1724,2026-04-10 23:40:00,2026-04-11 04:40:00,5915.0,17673.8,0.3,5443.7,30579.4
1725,2026-04-10 23:45:00,2026-04-11 04:45:00,5804.1,17664.5,0.1,5490.5,30776.8
1726,2026-04-10 23:50:00,2026-04-11 04:50:00,5755.5,17689.6,0.3,5394.4,30307.0
1727,2026-04-10 23:55:00,2026-04-11 04:55:00,5671.5,17657.9,0.3,5408.2,30240.6


### 23. GenMixHourly (spp_physical.GenMixHourly)

In [31]:
print("=== 23. GenMixHourly ===")
pd.set_option("display.max_columns", None)
for s, e, period in PERIODS:
    df_genmixhr = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_physical.GenMixHourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt, hr")
    check_df(df_genmixhr, period, False)
display(df_genmixhr.head())


=== 23. GenMixHourly ===
  [3/23-3/28] impute=False → Shape=(142, 33), Nulls=0, Zeros=1283
  [4/5-4/10] impute=False → Shape=(144, 33), Nulls=0, Zeros=1141


,dt,hr,CoalMarket,CoalSelf,CoalTotal,DieselFuelOilMarket,DieselFuelOilSelf,DieselFuelOilTotal,HydroMarket,HydroSelf,HydroTotal,NaturalGasMarket,NaturalGasSelf,NaturalGasTotal,NuclearMarket,NuclearSelf,NuclearTotal,SolarMarket,SolarSelf,SolarTotal,WasteDisposalServicesMarket,WasteDisposalServicesSelf,WasteDisposalServicesTotal,WindMarket,WindSelf,WindTotal,WasteHeatMarket,WasteHeatSelf,WasteHeatTotal,OtherMarket,OtherSelf,OtherTotal,AverageActualLoad
0,2026-04-05,0,3765.8,3311.4,7077.2,148.2,0.0,148.2,36.4,1521.5,1557.8,9930.8,1677.6,11608.5,0.0,2025.6,2025.6,0.0,0.3,0.3,0.0,6.2,6.2,0.0,7671.9,7671.9,0.0,0.0,0.0,2.5,52.3,54.9,29191.1
1,2026-04-05,1,3865.5,3339.7,7205.3,134.7,0.0,134.7,36.1,1477.9,1514.0,9530.4,1254.2,10784.6,0.0,2026.2,2026.2,0.0,0.3,0.3,0.0,6.2,6.2,0.0,7822.0,7822.0,0.0,0.0,0.0,0.0,82.5,82.5,28615.6
2,2026-04-05,2,3953.2,3300.5,7253.7,147.6,0.0,147.6,38.9,1457.3,1496.2,9429.3,1291.9,10721.2,0.0,2025.7,2025.7,0.0,0.3,0.3,0.0,6.2,6.2,0.0,7952.4,7952.4,0.0,0.0,0.0,49.9,27.5,77.4,28337.1
3,2026-04-05,3,4022.8,3387.1,7409.9,146.6,0.0,146.6,35.9,1470.9,1506.9,9846.7,1442.2,11288.9,0.0,2025.7,2025.7,0.0,0.3,0.3,0.0,6.3,6.3,0.0,7114.2,7114.2,0.0,0.0,0.0,33.4,27.5,60.9,28356.4
4,2026-04-05,4,4052.0,3534.2,7586.2,147.2,0.0,147.2,39.3,1519.8,1559.1,10139.0,1443.6,11582.5,0.0,2026.1,2026.1,0.0,0.2,0.2,0.0,6.3,6.3,0.0,6830.5,6830.5,0.0,0.0,0.0,30.1,27.4,57.5,28620.3


### 24. BAAZonalGenMix (spp_physical.BAAZonalGenMix) — E and W

In [32]:
print("=== 24. BAAZonalGenMix ===")
for s, e, period in PERIODS:
    df_baa_genmix = sql_functions.download_df_from_sql_db(
        f"SELECT IntervalEnd, BAA, CoalTotal, WindTotal, SolarTotal, NaturalGasTotal, AverageActualLoad "
        f"FROM spp_physical.BAAZonalGenMix "
        f"WHERE DATE(IntervalEnd) BETWEEN '{s}' AND '{e}' ORDER BY IntervalEnd DESC, BAA")
    check_df(df_baa_genmix, period, False)
df_baa_genmix.head(5)


=== 24. BAAZonalGenMix ===
  [3/23-3/28] impute=False → Shape=(1691, 7), Nulls=0, Zeros=47
  [4/5-4/10] impute=False → Shape=(3456, 7), Nulls=0, Zeros=613


,IntervalEnd,BAA,CoalTotal,WindTotal,SolarTotal,NaturalGasTotal,AverageActualLoad
0,2026-04-10 23:55:00,E,5033.7,16845.6,0.3,4958.4,27797.1
1,2026-04-10 23:55:00,W,637.8,812.3,0.0,449.8,2443.5
2,2026-04-10 23:50:00,E,5117.8,16877.9,0.3,4927.2,27851.3
3,2026-04-10 23:50:00,W,637.7,811.7,0.0,467.2,2455.7
4,2026-04-10 23:45:00,E,5166.5,16846.3,0.1,5023.3,28068.6


### 25. BAAZonalGenMixHourly (spp_physical.BAAZonalGenMixHourly) — E + W sum vs GenMixHourly

In [33]:
print("=== 25. BAAZonalGenMixHourly ===")
for s, e, period in PERIODS:
    df_baa_hr = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_physical.BAAZonalGenMixHourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt, hr, BAA")
    check_df(df_baa_hr, period, False)
df_baa_hr.head(5)


=== 25. BAAZonalGenMixHourly ===
  [3/23-3/28] impute=False → Shape=(142, 34), Nulls=0, Zeros=1283
  [4/5-4/10] impute=False → Shape=(288, 34), Nulls=0, Zeros=3294


,dt,hr,BAA,CoalMarket,CoalSelf,CoalTotal,DieselFuelOilMarket,DieselFuelOilSelf,DieselFuelOilTotal,HydroMarket,HydroSelf,HydroTotal,NaturalGasMarket,NaturalGasSelf,NaturalGasTotal,NuclearMarket,NuclearSelf,NuclearTotal,SolarMarket,SolarSelf,SolarTotal,WasteDisposalServicesMarket,WasteDisposalServicesSelf,WasteDisposalServicesTotal,WindMarket,WindSelf,WindTotal,WasteHeatMarket,WasteHeatSelf,WasteHeatTotal,OtherMarket,OtherSelf,OtherTotal,AverageActualLoad
0,2026-04-05,0,E,3589.6,2653.5,6243.0,0.0,0.0,0.0,28.2,1067.6,1095.8,9043.1,1572.8,10615.9,0.0,2025.6,2025.6,0.0,0.3,0.3,0.0,6.2,6.2,0.0,7432.4,7432.4,0.0,0.0,0.0,2.5,22.4,24.9,26830.5
1,2026-04-05,0,W,176.2,657.9,834.1,148.2,0.0,148.2,8.2,453.8,462.1,887.8,104.8,992.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,239.4,239.4,0.0,0.0,0.0,0.0,29.9,29.9,2360.6
2,2026-04-05,1,E,3708.0,2601.0,6308.9,0.0,0.0,0.0,27.9,1041.4,1069.3,8711.2,1183.3,9894.4,0.0,2026.2,2026.2,0.0,0.3,0.3,0.0,6.2,6.2,0.0,7679.7,7679.7,0.0,0.0,0.0,0.0,22.5,22.5,26377.3
3,2026-04-05,1,W,157.6,738.8,896.3,134.7,0.0,134.7,8.2,436.5,444.7,819.2,70.9,890.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,142.3,142.3,0.0,0.0,0.0,0.0,60.0,60.0,2238.3
4,2026-04-05,2,E,3761.1,2487.8,6248.9,0.0,0.0,0.0,27.8,1041.6,1069.5,8601.5,1233.9,9835.4,0.0,2025.7,2025.7,0.0,0.3,0.3,0.0,6.2,6.2,0.0,7895.8,7895.8,0.0,0.0,0.0,-10.0,22.5,12.5,26225.0


---
## TIE FLOWS

### 26. TieFlows (spp_physical.TieFlows)

In [34]:
print("=== 26. TieFlows ===")
for s, e, period in PERIODS:
    df_tieflow = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_physical.TieFlows "
        f"WHERE DATE(GMTTime) BETWEEN '{s}' AND '{e}' ORDER BY MarketTime LIMIT 200")
    check_df(df_tieflow, period, False)
print("Columns:", df_tieflow.columns.tolist())
df_tieflow.head(5)


=== 26. TieFlows ===
  [3/23-3/28] impute=False → Shape=(200, 30), Nulls=200, Zeros=817
  [4/5-4/10] impute=False → Shape=(0, 30), Nulls=0, Zeros=0.0
Columns: ['MarketTime', 'GMTTime', 'SPPNSI', 'DPC', 'MCWEST', 'CLEC', 'SPC', 'AMRN', 'OTP', 'GRE', 'MDU', 'SOUC', 'SPA', 'SGE', 'AECI', 'ERCOTN', 'WAUE', 'EDDY', 'ERCOTE', 'BLKW', 'LAMAR', 'MEC', 'SPPNAI', 'RCEAST', 'EES', 'NSP', 'SCSE', 'ALTW', 'TVA', 'SIKE']


,MarketTime,GMTTime,SPPNSI,DPC,MCWEST,CLEC,SPC,AMRN,OTP,GRE,MDU,SOUC,SPA,SGE,AECI,ERCOTN,WAUE,EDDY,ERCOTE,BLKW,LAMAR,MEC,SPPNAI,RCEAST,EES,NSP,SCSE,ALTW,TVA,SIKE


---
## RNU RATE

### 27. RNU Rate (spp_virtual.rnu_rate)

In [35]:
print("=== 27. RNU Rate ===")
for s, e, period in PERIODS:
    df_rnu = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_virtual.rnu_rate "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt")
    check_df(df_rnu, period, False)
df_rnu.head(10)


=== 27. RNU Rate ===
  [3/23-3/28] impute=False → Shape=(6, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(6, 5), Nulls=0, Zeros=0


,dt,Run_Type,RT_RNU_Dist_Rate,RNU_Rate_Forecast,source
0,2026-04-05,S7,0.0404,-0.4250,ftp
1,2026-04-06,S7,0.1962,-0.4250,ftp
2,2026-04-07,S7,-0.0982,-0.4250,ftp
3,2026-04-08,Frcst,-0.4086,-0.4086,ForecastV1
4,2026-04-09,Frcst,-0.4086,-0.4086,ForecastV1
5,2026-04-10,Frcst,-0.4086,-0.4086,ForecastV1


### 28. RNU Rate Detailed (spp_virtual.rnu_rate_detailed)

In [36]:
print("=== 28. RNU Rate Detailed ===")
for s, e, period in PERIODS:
    df_rnu_det = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_virtual.rnu_rate_detailed "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt")
    check_df(df_rnu_det, period, False)
df_rnu_det.head(10)


=== 28. RNU Rate Detailed ===
  [3/23-3/28] impute=False → Shape=(6, 14), Nulls=0, Zeros=2
  [4/5-4/10] impute=False → Shape=(3, 14), Nulls=21, Zeros=0


,auto_id,dt,Run_Type,DA_Inadequacy,RT_Inadequacy,RT_Out_of_Merit,RT_Reg_Dep_ADJ,RT_JOA,RT_Net_Inadvertent,RT_Congestion,RT_RNU_Dist_Qty,RT_RNU_Dist_Rate,RT_RNU_Hrly_Amt,source
0,9023,2026-04-05,S7,363921.0,-294816.0,None,None,None,None,None,None,0.0404,None,ftp
1,9022,2026-04-06,S7,387522.0,-14629.7,None,None,None,None,None,None,0.1962,None,ftp
2,9021,2026-04-07,S7,66097.2,-269056.0,None,None,None,None,None,None,-0.0982,None,ftp


---
## OP CHARGES (MWP)

### 29. Op Reserve and Fees (spp_virtual.v3_op_reserve_and_fees)

In [37]:
print("=== 29. Op Reserve and Fees ===")
for s, e, period in PERIODS:
    df_opcharge = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_virtual.v3_op_reserve_and_fees "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt")
    check_df(df_opcharge, period, False)
print("Columns:", df_opcharge.columns.tolist())
df_opcharge.head(10)


=== 29. Op Reserve and Fees ===
  [3/23-3/28] impute=False → Shape=(6, 5), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(9, 5), Nulls=0, Zeros=0
Columns: ['dt', 'BAA', 'daMwpDistRate', 'rtMwpDistRate', 'source']


,dt,BAA,daMwpDistRate,rtMwpDistRate,source
0,2026-04-05,E,0.0356,0.3619,ftp
1,2026-04-05,W,0.0226,0.3614,ftp
2,2026-04-06,E,0.2741,1.5644,ftp
3,2026-04-06,W,0.0873,1.9358,ftp
4,2026-04-07,E,0.8237,1.7262,ftp
5,2026-04-07,W,0.0372,4.1294,ftp
6,2026-04-08,E,0.4933,0.7061,forecastV2
7,2026-04-09,E,0.4124,1.0509,forecastV2
8,2026-04-10,E,0.2623,0.9217,forecastV2


---
## CONSTRAINTS

### 30. DA Constraint Congestion (spp_constraints.congestion_da_hourly_v3)

Raw table — downloaded by `SPP_download_DA_Constraints_v20170324.php`. Tracks DA binding constraint shadow prices (M-values) per hour.

In [38]:
print("=== 30. DA Constraint Congestion (raw) ===")
for s, e, period in PERIODS:
    df_da_cong = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_constraints.congestion_da_hourly_v3 "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY mvalue LIMIT 40")
    check_df(df_da_cong, period, False)
display(df_da_cong.head(10))


=== 30. DA Constraint Congestion (raw) ===
  [3/23-3/28] impute=False → Shape=(40, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(40, 4), Nulls=0, Zeros=0


,dt,hr,ConstraintNum,MValue
0,2026-04-08,20,774237,-1431.1290
1,2026-04-06,19,868929,-761.6196
2,2026-04-06,20,868929,-760.6649
3,2026-04-06,18,868929,-669.4581
4,2026-04-06,17,868929,-621.4143
5,2026-04-06,15,868929,-608.7178
6,2026-04-06,16,868929,-607.1592
7,2026-04-06,14,868929,-603.3156
8,2026-04-06,8,868929,-578.9339
9,2026-04-06,21,868929,-559.4016


### 31. DA Constraint M-Values via nighthawk `Constraint.get_mvalues()`

Reads from `congestion_da_hourly_v3`. Returns M-values (shadow prices × MW) for all constraints. Use `granularity='hourly'` for hour-by-hour breakdown.

In [39]:
from nighthawk.data.network.constraint import Constraint
constraint_spp = Constraint(oops_constraint_num_df=None, market="SPP")
print("=== 31. Constraint.get_mvalues ===")
for s, e, period in PERIODS:
    for ctype in ["DA", "RT"]:
        for gran in ["hourly", "daily"]:
            df_mv = constraint_spp.get_mvalues(s, e, type=ctype, granularity=gran)
            null_count = df_mv.isnull().sum().sum()
            zero_count = (df_mv.select_dtypes("number") == 0).sum().sum()
            print(f"  [{period}] type={ctype}, granularity={gran} → Shape={df_mv.shape}, Nulls={null_count}, Zeros={zero_count}")
display(df_mv.sort_values("mvalue", ascending=False).head(10))


=== 31. Constraint.get_mvalues ===
  [3/23-3/28] type=DA, granularity=hourly → Shape=(2490, 4), Nulls=0, Zeros=0
  [3/23-3/28] type=DA, granularity=daily → Shape=(370, 3), Nulls=0, Zeros=0
  [3/23-3/28] type=RT, granularity=hourly → Shape=(924, 4), Nulls=0, Zeros=0
  [3/23-3/28] type=RT, granularity=daily → Shape=(203, 3), Nulls=0, Zeros=0
  [4/5-4/10] type=DA, granularity=hourly → Shape=(2574, 4), Nulls=0, Zeros=0
  [4/5-4/10] type=DA, granularity=daily → Shape=(332, 3), Nulls=0, Zeros=0
  [4/5-4/10] type=RT, granularity=hourly → Shape=(1613, 4), Nulls=0, Zeros=0
  [4/5-4/10] type=RT, granularity=daily → Shape=(248, 3), Nulls=0, Zeros=0


,oops_constraint_num,dt,mvalue
66,560051,2026-04-08,-0.2112
229,869476,2026-04-07,-0.8572
144,731146,2026-04-08,-1.0540
45,532937,2026-04-10,-1.1273
103,658026,2026-04-07,-2.2459
0,32565,2026-04-05,-2.3767
238,869636,2026-04-08,-2.5807
230,869476,2026-04-08,-2.6478
38,483841,2026-04-06,-2.6790
226,869424,2026-04-07,-3.1262


### 32. DA Constraint Family M-Values via `ConstraintFamily.get_mvalues()`

Reads `congestion_da_hourly_v3` + `ConstraintFamilyMaster_v3`. Groups constraints by monitored facility name.

In [40]:
from nighthawk.data.network.constraint_family import ConstraintFamily
cf_spp = ConstraintFamily(constraint_family_num_df=None, opexchange="SPP")
print("=== 32. ConstraintFamily.get_mvalues ===")
for s, e, period in PERIODS:
    for ctype in ["DA", "RT"]:
        for gran in ["daily", "hourly"]:
            for summary in [False, True]:
                df_cf = cf_spp.get_mvalues(s, e, type=ctype, granularity=gran, summary=summary)
                null_count = df_cf.isnull().sum().sum()
                zero_count = (df_cf.select_dtypes("number") == 0).sum().sum()
                print(f"  [{period}] type={ctype}, gran={gran}, summary={summary} → Shape={df_cf.shape}, Nulls={null_count}, Zeros={zero_count}")
display(df_cf.sort_values("mvalue", ascending=False).head(10))


=== 32. ConstraintFamily.get_mvalues ===
  [3/23-3/28] type=DA, gran=daily, summary=False → Shape=(346, 3), Nulls=0, Zeros=0
  [3/23-3/28] type=DA, gran=daily, summary=True → Shape=(140, 2), Nulls=0, Zeros=0
  [3/23-3/28] type=DA, gran=hourly, summary=False → Shape=(2480, 4), Nulls=0, Zeros=0
  [3/23-3/28] type=DA, gran=hourly, summary=True → Shape=(140, 2), Nulls=0, Zeros=0
  [3/23-3/28] type=RT, gran=daily, summary=False → Shape=(193, 3), Nulls=0, Zeros=0
  [3/23-3/28] type=RT, gran=daily, summary=True → Shape=(76, 2), Nulls=0, Zeros=0
  [3/23-3/28] type=RT, gran=hourly, summary=False → Shape=(909, 4), Nulls=0, Zeros=0
  [3/23-3/28] type=RT, gran=hourly, summary=True → Shape=(76, 2), Nulls=0, Zeros=0
  [4/5-4/10] type=DA, gran=daily, summary=False → Shape=(303, 3), Nulls=0, Zeros=0
  [4/5-4/10] type=DA, gran=daily, summary=True → Shape=(133, 2), Nulls=0, Zeros=0
  [4/5-4/10] type=DA, gran=hourly, summary=False → Shape=(2544, 4), Nulls=0, Zeros=0
  [4/5-4/10] type=DA, gran=hourly, sum

,constraintFamilyNum,mvalue
14,441,-0.2112
97,1887984,-2.5807
71,785093,-2.6790
61,490699,-3.1262
93,1792291,-3.5050
86,1492236,-4.5274
91,1735779,-4.7564
87,1557556,-5.6024
38,6299,-5.6536
39,14969,-6.6395


### 33. RT Constraint Congestion Hourly (spp_constraints.congestion_rt_hourly_v3)

Aggregated from 5-min intervals. Same structure as DA but for RTBM.

In [41]:
print("=== 33. RT Constraint Congestion Hourly (raw) ===")
for s, e, period in PERIODS:
    df_rt_cong_hr = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_constraints.congestion_rt_hourly_v3 "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY mvalue LIMIT 40")
    check_df(df_rt_cong_hr, period, False)
display(df_rt_cong_hr.head(10))


=== 33. RT Constraint Congestion Hourly (raw) ===
  [3/23-3/28] impute=False → Shape=(40, 4), Nulls=0, Zeros=0
  [4/5-4/10] impute=False → Shape=(40, 4), Nulls=0, Zeros=0


,dt,hr,ConstraintNum,MValue
0,2026-04-07,2,862358,-1444.9458
1,2026-04-10,2,869563,-1433.3003
2,2026-04-10,3,869563,-1410.6919
3,2026-04-09,24,869563,-1406.9004
4,2026-04-10,1,869563,-1395.2299
5,2026-04-05,23,433608,-1381.3643
6,2026-04-09,22,869563,-1334.2565
7,2026-04-07,1,862358,-1331.8623
8,2026-04-09,23,869563,-1331.6665
9,2026-04-09,20,625307,-1266.1617


### 34. RT Constraint Congestion 5-min (spp_constraints.congestion_rt_5_min_v3)

Highest-frequency constraint data — used by the contour map for real-time display.

In [ ]:
print("=== 34. RT Constraint 5-min (from hourly table) ===")
for s, e, period in PERIODS:
    df_rt_5m = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_constraints.congestion_rt_hourly_v3 "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt LIMIT 40")
    check_df(df_rt_5m, period, False)
display(df_rt_5m.head(10))


### 35. RT Constraint M-Values via nighthawk `Constraint.get_mvalues(type='RT')`

In [ ]:
print("=== 35. Constraint.get_mvalues RT ===")
for s, e, period in PERIODS:
    df_rt_mval = constraint_spp.get_mvalues(s, e, type="RT", granularity="hourly")
    check_df(df_rt_mval, period, False, extra_label="RT hourly")
display(df_rt_mval.sort_values("mvalue", ascending=False).head(10))


### 36. Constraint Rolling Summaries (7D / 30D / 365D)

Pre-computed rolling M-value aggregates for trend analysis.

In [ ]:
print("=== 36. Constraint Rolling Summaries ===")
for tbl, label in [("congestion_rt_7D_v3","7D"), ("congestion_rt_30D_v3","30D"), ("congestion_rt_365D_v3","365D")]:
    df_roll = sql_functions.download_df_from_sql_db(f"SELECT * FROM spp_constraints.{tbl} LIMIT 10")
    check_df(df_roll, label, False)
    display(df_roll)


---
## LMP (SETTLEMENT LOCATION PRICES)

**Nighthawk entry point:** `Node(node_nums=[...], market='SPP').get_price()`  
**Source tables:**
- DA hourly → `spp_lmp.settlement_location_da_hourly` (`da_value`, `congestional_value`, `marginalloss_value`)
- RT hourly → `spp_lmp.settlement_location_rt_hourly` (`rt_value`, `congestional_value`, `marginalloss_value`)
- RT daily summary → `spp_lmp.sl_rt_daily_summary` (`daily`, `on_peak`, `off_peak`, `daily_congestion`, ...)
- DA daily summary → `spp_lmp.sl_da_daily_summary`

**Note:** `rt_hrly_from_5min_table` for SPP points to the same `settlement_location_rt_hourly` (no separate fallback table).  
**Reprice:** `settlement_location_da_hourly_reprice` is written weekly; 0 rows for 2026-04-01 is expected.

### 37. DA Hourly LMP — Raw (spp_lmp.settlement_location_da_hourly)

Written by `download_LMP_from_CSV_to_DB3.php` via `REPLACE INTO`.  
**Columns:** `dt, hr, node_num, da_value (LMP), congestional_value (MCC), marginalloss_value (MLC)`  
**Expected for 2026-04-01:** 1580 nodes × 24 hrs = **37,920 rows**

In [ ]:
HUB_NODE = 635
print("=== 37. DA Hourly LMP (raw) ===")
for s, e, period in PERIODS:
    df_da_lmp = sql_functions.download_df_from_sql_db(
        f"SELECT dt, hr, node_num, da_value, congestional_value, marginalloss_value "
        f"FROM spp_lmp.settlement_location_da_hourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt, hr")
    df_da_lmp["implied_slack"] = df_da_lmp["da_value"] - df_da_lmp["congestional_value"] - df_da_lmp["marginalloss_value"]
    check_df(df_da_lmp, period, False)
display(df_da_lmp.head())


### 38. Row count check — DA 2026-04-01 should have exactly 37,920 rows

In [ ]:
print("=== 38. DA Row Count ===")
for s, e, period in PERIODS:
    df_da_cnt = sql_functions.download_df_from_sql_db(
        f"SELECT dt, COUNT(DISTINCT node_num) as nodes, COUNT(DISTINCT hr) as hrs, COUNT(*) as total "
        f"FROM spp_lmp.settlement_location_da_hourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' GROUP BY dt ORDER BY dt")
    check_df(df_da_cnt, period, False)
    display(df_da_cnt)


### 39. DA LMP Reprice (spp_lmp.settlement_location_da_hourly_reprice)

Written by `spp_dataDownload_da_lmp_reprice.py` — runs **weekly**, not daily.  
0 rows for a given date is **normal** unless that date was repriced.  
Columns: `dt, hr, node_num, Revision, Reason, da_value, marginalloss_value, congestional_value, Modified_Date`

In [ ]:
print("=== 39. DA LMP Reprice ===")
for s, e, period in PERIODS:
    df_reprice = sql_functions.download_df_from_sql_db(
        f"SELECT dt, hr, node_num, Revision, Reason, da_value, Modified_Date "
        f"FROM spp_lmp.settlement_location_da_hourly_reprice "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt, hr, Revision")
    check_df(df_reprice, period, False)
print("(0 rows expected for most dates — reprice is weekly)")
display(df_reprice)


### 40. RT 5-min LMP — Raw (spp_lmp.sl_rt_5_min)

Written by `SPP_download_DART_LMP.php` (mode=rt) and gap-filled by `SPPLMPDailyCheck_5min.py`.  
**Columns:** `forecast_time (interval end datetime), dt, hr, node_num, rt_value, congestional_value, marginalloss_value`  
**Expected:** 12 intervals per hr × 24 hrs × ~1580 nodes

In [ ]:
print("=== 40. RT 5-min LMP (raw) ===")
for s, e, period in PERIODS:
    df_rt_5m_lmp = sql_functions.download_df_from_sql_db(
        f"SELECT forecast_time, dt, hr, node_num, rt_value, congestional_value, marginalloss_value "
        f"FROM spp_lmp.sl_rt_5_min "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY forecast_time")
    check_df(df_rt_5m_lmp, period, False)
display(df_rt_5m_lmp.head())


### 41. RT Hourly LMP — Raw (spp_lmp.settlement_location_rt_hourly)

Derived from `sl_rt_5_min` by `AVG(rt_value) GROUP BY (dt, hr, node_num)`.  
**Expected for 2026-04-01:** up to 37,920 rows — actual is **37,622** (298 gaps expected due to RTBM timing)

In [ ]:
print("=== 41. RT Hourly LMP (raw) ===")
for s, e, period in PERIODS:
    df_rt_lmp = sql_functions.download_df_from_sql_db(
        f"SELECT dt, hr, node_num, rt_value, congestional_value, marginalloss_value "
        f"FROM spp_lmp.settlement_location_rt_hourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt, hr")
    check_df(df_rt_lmp, period, False)
display(df_rt_lmp.head())


### 42. Node.get_price() — nighthawk function (DA + RT hourly)

**SPP DA path:** reads `settlement_location_da_hourly` directly  
**SPP RT path:** reads `settlement_location_rt_hourly`; if end_dt > max(dt), falls back to same table  
**Column mapping:**
- `da_value` → `da_total` (LMP)
- `congestional_value` → `da_congestion` (MCC)
- `marginalloss_value` → `da_loss` (MLC)
- `da_value - congestional_value - marginalloss_value` → `da_slack`

In [ ]:
from nighthawk.data.network.node import Node
node_spp = Node(node_nums=[635], market="SPP")
print("=== 42. Node.get_price ===")
for s, e, period in PERIODS:
    for ctype in [["DA"], ["RT"], ["DA", "RT"]]:
        for gran in ["hourly", "daily"]:
            df_price = node_spp.get_price(s, e, component=["LMP","MCC","MLC","Slack"], type=ctype, granularity=gran)
            null_count = df_price.isnull().sum().sum()
            zero_count = (df_price.select_dtypes("number") == 0).sum().sum()
            print(f"  [{period}] type={ctype}, gran={gran} → Shape={df_price.shape}, Nulls={null_count}, Zeros={zero_count}")
            if null_count > 0:
                null_mask = df_price.isnull().any(axis=1)
                print(f"  ⚠️  WARNING: {null_count} nulls in get_price!")
                display(df_price[null_mask].head(5))
display(df_price.head())


### 43. Cross-check: Node.get_price() vs raw table — values must match

In [ ]:
print("=== 43. Cross-check Node.get_price vs raw ===")
for s, e, period in PERIODS:
    df_da_raw = sql_functions.download_df_from_sql_db(
        f"SELECT dt, hr, congestional_value FROM spp_lmp.settlement_location_da_hourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt, hr")
    df_rt_raw = sql_functions.download_df_from_sql_db(
        f"SELECT dt, hr, rt_value FROM spp_lmp.settlement_location_rt_hourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt, hr")
    df_nd = node_spp.get_price(s, e, component=["LMP","MCC","MLC","Slack"], type=["DA","RT"], granularity="hourly")
    df_da_raw["dt"] = df_da_raw["dt"].astype(str)
    df_nd["dt"] = df_nd["dt"].astype(str)
    m_da = df_da_raw[["dt","hr","congestional_value"]].merge(df_nd[["dt","hr","da_congestion"]], on=["dt","hr"], how="outer")
    m_da["diff"] = (m_da["congestional_value"] - m_da["da_congestion"]).abs()
    print(f"  [{period}] DA MCC max diff: {m_da["diff"].max():.4f}")
    m_rt = df_rt_raw[["dt","hr","rt_value"]].merge(df_nd[["dt","hr","rt_total"]], on=["dt","hr"], how="outer")
    m_rt["diff"] = (m_rt["rt_value"] - m_rt["rt_total"]).abs()
    print(f"  [{period}] RT LMP max diff: {m_rt["diff"].max():.4f}")


### 44. Last-30-Day Rolling Tables (sl_da_hourly_last30days / sl_rt_hourly_last30days)

Same schema as core hourly tables. Used by nighthawk for faster recent-date queries.  
Written by OOPSify after each DA/RT download.

In [ ]:
print("=== 44. Last-30-Day Rolling Tables ===")
for s, e, period in PERIODS:
    df_da_30d = sql_functions.download_df_from_sql_db(
        f"SELECT dt, hr, node_num, da_value, congestional_value "
        f"FROM spp_lmp.sl_da_hourly_last30days "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt, hr")
    df_rt_30d = sql_functions.download_df_from_sql_db(
        f"SELECT dt, hr, node_num, rt_value, congestional_value "
        f"FROM spp_lmp.sl_rt_hourly_last30days "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt, hr")
    check_df(df_da_30d, period, False, extra_label="DA last30d")
    check_df(df_rt_30d, period, False, extra_label="RT last30d")
display(df_da_30d.head())


### 45. Daily Summary LMP (sl_rt_daily_summary / sl_da_daily_summary)

Written by OOPSify. Used by `Node.get_price(..., granularity='daily')`.  
**Columns:** `dt, node_num, daily, on_peak, off_peak, daily_congestion, on_peak_congestion, off_peak_congestion, daily_marginloss, ...`  
**Expected for 2026-04-01:** 1,580 rows (one per node)

In [ ]:
print("=== 45. Daily Summary LMP ===")
for s, e, period in PERIODS:
    df_rt_daily = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_lmp.sl_rt_daily_summary "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt")
    check_df(df_rt_daily, period, False, extra_label="RT daily node635")
    df_da_daily = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_lmp.sl_da_daily_summary "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND node_num=635 ORDER BY dt")
    check_df(df_da_daily, period, False, extra_label="DA daily node635")
display(df_rt_daily)


### 46. Node.get_price() daily granularity — reads sl_rt_daily_summary

In [ ]:
print("=== 46. Node.get_price daily ===")
for s, e, period in PERIODS:
    for gran in ["daily", "hourly"]:
        df_node_daily = node_spp.get_price(s, e, component=["LMP","MCC","MLC"], type=["DA","RT"], granularity=gran)
        null_count = df_node_daily.isnull().sum().sum()
        zero_count = (df_node_daily.select_dtypes("number") == 0).sum().sum()
        print(f"  [{period}] gran={gran} → Shape={df_node_daily.shape}, Nulls={null_count}, Zeros={zero_count}")
display(df_node_daily.head())


### 47. RT 5-min Slack Last 2 Years — settlement_location_rt_5min_slack_last2years
> Script `spp_dataDownload_rt_slack_last2years.py` is **deprecated** (2026-03-12); table is now refreshed by `spp_update_lmp_summary.py` (`update_rt_5min_slack_last2years`).

In [ ]:
print("=== 47. RT 5-min Slack Last 2 Years ===")
for s, e, period in PERIODS:
    df_rt_slack = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_lmp.settlement_location_rt_5min_slack_last2years "
        f"WHERE dt BETWEEN '{s}' AND '{e}' AND hr=1")
    check_df(df_rt_slack, period, False)
df_rt_slack_cnt = sql_functions.download_df_from_sql_db(
    "SELECT COUNT(*) as total, MIN(dt) as min_dt, MAX(dt) as max_dt "
    "FROM spp_lmp.settlement_location_rt_5min_slack_last2years")
display(df_rt_slack_cnt)


### 48. Multi-Day LMP Forecast — settlement_location_multi_day_forecast

In [ ]:
print("=== 48. Multi-Day LMP Forecast ===")
df_mdfc = sql_functions.download_df_from_sql_db(
    "SELECT * FROM spp_lmp.settlement_location_multi_day_forecast "
    "ORDER BY as_of_dt DESC, dt, hr LIMIT 10")
check_df(df_mdfc, "recent", False)
df_mdfc_virt = sql_functions.download_df_from_sql_db(
    "SELECT * FROM spp_lmp.settlement_location_multi_day_forecast_virtual "
    "ORDER BY dt DESC, hr LIMIT 10")
check_df(df_mdfc_virt, "recent", False, extra_label="virtual")
display(df_mdfc); display(df_mdfc_virt)


### 49. DA MCP Zonal Hourly — reserve_mcp_da_zonal_hourly

In [ ]:
print("=== 49. DA MCP Zonal Hourly ===")
for s, e, period in PERIODS:
    df_da_mcp = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_lmp.reserve_mcp_da_zonal_hourly "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt, hr, Reserve_Zone")
    check_df(df_da_mcp, period, False)
df_da_mcp_cnt = sql_functions.download_df_from_sql_db(
    "SELECT COUNT(*) as total, MIN(dt) as min_dt, MAX(dt) as max_dt "
    "FROM spp_lmp.reserve_mcp_da_zonal_hourly")
display(df_da_mcp_cnt); display(df_da_mcp.head())


### 50. DA Market Clearing — spp_virtual.market_clearing

In [ ]:
print("=== 50. DA Market Clearing ===")
for s, e, period in PERIODS:
    df_mc = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_virtual.market_clearing "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt, hr")
    check_df(df_mc, period, False)
display(df_mc.head())


### 51. DA Virtual Clearing by MOA — spp_virtual.moa_virtual_clearing

In [ ]:
print("=== 51. MOA Virtual Clearing ===")
for s, e, period in PERIODS:
    df_moa = sql_functions.download_df_from_sql_db(
        f"SELECT * FROM spp_virtual.moa_virtual_clearing "
        f"WHERE dt BETWEEN '{s}' AND '{e}' ORDER BY dt, hr, moa")
    check_df(df_moa, period, False)
display(df_moa.head())


### 52. Hourly Available Outage Maintenance Margin — `get_data_and_mapping_for_avail_maintenance_margin()`
> Source tables: `spp_physical.hourly_available_outage_maintenance_margin_virtual` (forecast) and `hourly_available_outage_maintenance_margin_latest` (actual). SWPW tables not used in nighthawk.

In [ ]:
from nighthawk.data.pipeline.var_handler.avail_maint_margin import get_data_and_mapping_for_avail_maintenance_margin
print("=== 52. Avail Maintenance Margin ===")
for s, e, period in PERIODS:
    for impute in [True, False]:
        data_amm, mapping_amm = get_data_and_mapping_for_avail_maintenance_margin(
            opexchange="SPP", start_dt=s, end_dt=e, var_spec=["f", "a"], impute=impute)
        check_df(data_amm, period, impute)
# Extra: intp_method note — only "linear" is supported
print("  intp_method: only linear supported, default used")
print("columns:", data_amm.columns.tolist())
display(data_amm.head()); display(mapping_amm)
